Project Setup

In [ ]:
print("PRODUCT MATCHING SYSTEM FOR E-COMMERCE ELECTRONICS")
print("FINAL YEAR PROJECT - BSc (Hons) COMPUTER SCIENCE")
print("UNIVERSITY OF GREENWICH")


#Installing the required packages
print("\n Installing required packages...")
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly
!pip install -q python-Levenshtein sentence-transformers transformers
!pip install -q torch torchvision tensorflow
!pip install -q nltk tqdm openpyxl
!pip install -q umap-learn scikit-plot
!pip install -q optuna  #For hyperparameter optimisation

print("Package installation complete!")

Import libraries and mount Google Drive

In [ ]:
#Mount Google Drive to access data files
from google.colab import drive
drive.mount('/content/drive')

#Core Python libraries
import os
import json
import random
import warnings
import time
from datetime import datetime
from typing import Tuple, List, Dict, Any, Optional, Union
from dataclasses import dataclass
from collections import defaultdict

#Data manipulation and numerical computing
import numpy as np
import pandas as pd

#Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

#Text processing and NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import Levenshtein

#Machine learning with scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier

#Deep learning with PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

#Sentence transformers for Siamese Network
from sentence_transformers import SentenceTransformer

#Progress bars
from tqdm.notebook import tqdm

#Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

#Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#Configure matplotlib for better visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
rcParams['figure.figsize'] = (12, 8)
rcParams['font.size'] = 12
rcParams['axes.titlesize'] = 16
rcParams['axes.labelsize'] = 14

print("Libraries imported successfully!")
print(f"PyTorch device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Configuration and setup

In [ ]:
#Configuration class for the product matching project
#Contains all parameters, paths, and settings
@dataclass
class ProjectConfig:
    #Project paths for organising files and outputs
    PROJECT_PATH: str = '/content/drive/MyDrive/FYP' #Root project directory
    DATA_PATH: str = os.path.join(PROJECT_PATH, 'data') #Directory for data files
    MODELS_PATH: str = os.path.join(PROJECT_PATH, 'models') #Directory for saved models
    RESULTS_PATH: str = os.path.join(PROJECT_PATH, 'results') #Directory for results
    VISUALISATIONS_PATH: str = os.path.join(PROJECT_PATH, 'visualisations') #Directory for plots

    #Data file names
    #This project uses the DeepMatcher Amazon-Walmart Electronics Dataset
    #tableA.csv: Amazon product listings (raw data)
    #tableB.csv: Walmart product listings (raw data)
    #matches.csv: Matches from the two datasets
    #Source: https://github.com/anhaidgroup/deepmatcher/blob/master/Datasets.md
    #Direct download: https://pages.cs.wisc.edu/~anhai/data1/deepmatcher_data/Structured/Walmart-Amazon/

    TABLE_A_FILE: str = 'tableA.csv' #First data set
    TABLE_B_FILE: str = 'tableB.csv' #Second data set

    #Data split proportions
    TEST_SIZE: float = 0.15    #15% for testing
    VALIDATION_SIZE: float = 0.15  #15% for validation
    TRAIN_SIZE: float = 0.70   #70% for training

    #Random Forest parameters
    RF_N_ESTIMATORS: int = 300      #Number of trees in the forest
    RF_MAX_DEPTH: int = 25          #Maximum depth of trees (to prevent overfitting)
    RF_MIN_SAMPLES_SPLIT: int = 5   #Minimum samples required to split a node
    RF_MIN_SAMPLES_LEAF: int = 2    #Minimum samples at leaf node

    #Siamese Network parameters
    SIAMESE_BATCH_SIZE: int = 64           #Batch size for training
    SIAMESE_LEARNING_RATE: float = 5e-5    #Learning rate
    SIAMESE_EPOCHS: int = 25               #Number of training epochs
    SIAMESE_EMBEDDING_DIM: int = 384       #Sentence-BERT embedding dimension
    SIAMESE_HIDDEN_DIM: int = 512          #Number of neurons in hidden layers

    #Sentence Transformer model (using all-MiniLM-L6-v2 for efficiency)
    SENTENCE_TRANSFORMER_MODEL: str = 'all-MiniLM-L6-v2'

    #Feature engineering settings to toggle different feature types
    USE_TEXT_SIMILARITY: bool = True   #Include text similarity features
    USE_CATEGORICAL_FEATURES: bool = True  #Include categorical features
    USE_NUMERICAL_FEATURES: bool = True    #Include numerical features

    #Text processing settings for cleaning and normalisation
    REMOVE_STOPWORDS: bool = True      #Remove common stopwords
    USE_STEMMING: bool = False         #Use stemming (False = use lemmatisation instead)
    MAX_TEXT_LENGTH: int = 512         #Maximum text length for processing

    #Evaluation metrics
    EVALUATION_METRICS: Tuple = ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')

    #Random seed for reproducibility which ensures same results each run
    RANDOM_STATE: int = SEED

    #Dataset creation
    NEGATIVE_TO_POSITIVE_RATIO: float = 1.0  #Ratio of negative to positive samples (match and non match)
    MAX_SYNTHETIC_PAIRS: int = 2000          #Maximum synthetic pairs to create

    #Hyperparameter tuning settings
    TUNE_RANDOM_FOREST: bool = True          #Enable Random Forest hyperparameter tuning
    TUNE_SIAMESE_NETWORK: bool = True        #Enable Siamese Network hyperparameter tuning

    #Random Forest tuning parameters
    RF_TUNE_CV_FOLDS: int = 5                #Number of cross-validation folds for tuning
    RF_TUNE_N_JOBS: int = -1                 #Use all available cores for tuning

    #Siamese Network tuning parameters
    SIAMESE_TUNE_EPOCHS: int = 10            #Number of epochs per trial for tuning
    SIAMESE_TUNE_TRIALS: int = 10            #Maximum number of trials for tuning
    SIAMESE_TUNE_TIMEOUT: int = 25200         #Maximum time for tuning in seconds (7 hours)


#Initialise configuration
config = ProjectConfig()

#Create project directories
print("Creating project directory structure...")
for path in [config.PROJECT_PATH, config.DATA_PATH, config.MODELS_PATH,
             config.RESULTS_PATH, config.VISUALISATIONS_PATH]:
    os.makedirs(path, exist_ok=True) #Create directory, ignore if already exists
    print(f"Created: {path}")

print("Project configuration completed!")

Data loader class

In [ ]:
#This class handles loading, inspection, and basic analysis of product data
#Also responsible for reading CSV files and providing data summaries
class ProductDataLoader:

    #Initialise the data loader with project configuration
    def __init__(self, config: ProjectConfig):

        self.config = config
        self.table_a = None #Will hold DataFrame for table A
        self.table_b = None #Will hold DataFrame for table B

    def load_product_tables(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        #Load the two product tables from CSV files
        #This returns tuple containing (table_a, table_b) DataFrames
        #Raises a FileNotFoundError: If data files are not found
        print("Loading product data tables")

        #Construct file paths by combining directory and filename
        table_a_path = os.path.join(self.config.DATA_PATH, self.config.TABLE_A_FILE)
        table_b_path = os.path.join(self.config.DATA_PATH, self.config.TABLE_B_FILE)

        #Check if the files exist
        if not os.path.exists(table_a_path):
            raise FileNotFoundError(f"Table A file not found: {table_a_path}")
        if not os.path.exists(table_b_path):
            raise FileNotFoundError(f"Table B file not found: {table_b_path}")

        #Load CSV files into pandas DataFrames
        try:
            self.table_a = pd.read_csv(table_a_path, encoding='utf-8')
            self.table_b = pd.read_csv(table_b_path, encoding='utf-8')

            print(f"Successfully loaded:")
            print(f"Table A: {len(self.table_a)} products, {len(self.table_a.columns)} columns")
            print(f"Table B: {len(self.table_b)} products, {len(self.table_b.columns)} columns")

            return self.table_a, self.table_b

        except Exception as e:
            print(f"Error loading data files: {str(e)}") #Display error message
            raise

    #Perform comprehensive data quality inspection
    #Returns a dictionary containing data quality metrics
    def inspect_data_quality(self) -> Dict[str, Any]:
        print("\nInspecting data quality...")

        quality_report = {
            'table_a': {},
            'table_b': {},
            'comparison': {}
        }

        #analyse each table seperately
        for table_name, table in [('table_a', self.table_a), ('table_b', self.table_b)]:
            print(f"\n {table_name.upper()} Quality Report:")

            #basic statistics
            n_rows = len(table) #Get the dataframe dimensions
            n_cols = len(table.columns) #Get the dataframe dimensions
            print(f"Dimensions: {n_rows} rows × {n_cols} columns")

            #Missing values analysis, count and percentage per column
            missing_counts = table.isnull().sum()
            missing_percentage = (missing_counts / n_rows * 100).round(2) #converting the count to a %

            #Store in report dictionary
            quality_report[table_name]['dimensions'] = (n_rows, n_cols)
            quality_report[table_name]['missing_counts'] = missing_counts.to_dict()
            quality_report[table_name]['missing_percentage'] = missing_percentage.to_dict()

            #print the columns with missing values
            print("\nMissing values by column:")
            for col in table.columns:
                missing = missing_counts[col]
                if missing > 0:
                    print(f"  {col}: {missing} ({missing_percentage[col]}%)")

            #Show data types of each column
            print("\n Data types:")
            for col, dtype in table.dtypes.items():
                print(f"  {col}: {dtype}")

        #Compare the two tables to find common and unique columns
        common_cols = set(self.table_a.columns) & set(self.table_b.columns)
        quality_report['comparison']['common_columns'] = list(common_cols)

        print(f"\n TABLES COMPARISON:")
        print(f"Common columns: {len(common_cols)}")
        print(f"Table A specific columns: {set(self.table_a.columns) - common_cols}")
        print(f"Table B specific columns: {set(self.table_b.columns) - common_cols}")

        return quality_report

    #Display sample products from both tables
    #n_samples the number of samples to display from each table
    def display_sample_products(self, n_samples: int = 5):
        print(f"\nSAMPLE PRODUCTS (first {n_samples} from each table):")

        print("\nTABLE A SAMPLES:")
        print("-" * 40)
        display(self.table_a.head(n_samples)) #Using Colab's display function

        print("\nTABLE B SAMPLES:")
        print("-" * 40)
        display(self.table_b.head(n_samples))

    #Calculate descriptive statistics for numerical columns
    #DataFrame with statistics for each numerical column is returned
    def get_column_statistics(self) -> pd.DataFrame:
        print("\n COLUMN STATISTICS:")

        #Combine both tables for comprehensive statistics
        all_tables = pd.concat([self.table_a, self.table_b], ignore_index=True)

        #Select numerical columns
        numerical_cols = all_tables.select_dtypes(include=[np.number]).columns.tolist()

        if not numerical_cols:
            print("No numerical columns found in the data.")
            return pd.DataFrame()

        #Create empty DataFrame to store statistics
        stats_df = pd.DataFrame(index=numerical_cols)
        #Calculate various statistics for each numerical column
        for col in numerical_cols:
            stats_df.loc[col, 'mean'] = all_tables[col].mean() #Average value
            stats_df.loc[col, 'std'] = all_tables[col].std() #Standard deviation
            stats_df.loc[col, 'min'] = all_tables[col].min() #minimum value
            stats_df.loc[col, '25%'] = all_tables[col].quantile(0.25) #25th percentile
            stats_df.loc[col, '50%'] = all_tables[col].quantile(0.50) #Median (50th percentile)
            stats_df.loc[col, '75%'] = all_tables[col].quantile(0.75) #75th percentile
            stats_df.loc[col, 'max'] = all_tables[col].max() #Maximum value
            stats_df.loc[col, 'missing'] = all_tables[col].isnull().sum() #Count of missing values
            stats_df.loc[col, 'missing_pct'] = (all_tables[col].isnull().sum() / len(all_tables) * 100).round(2) #The percentage missing

        #Display the statistics
        display(stats_df.round(4)) #to 4 decimal places

        return stats_df

#Initialise data loader
data_loader = ProductDataLoader(config)
print("Data loader initialised successfully!")

Text Preprocesser class

In [ ]:

#Download required NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt', quiet=True) #Tokenizer models
nltk.download('stopwords', quiet=True) #Common stopwords list
nltk.download('wordnet', quiet=True) #WordNet database for lemmatisation
nltk.download('omw-eng', quiet=True) #Open multilingual WordNet
print("NLTK resources downloaded!")

#Handles text preprocessing and normalisation for product descriptions
#Includes cleaning, stopword removal, lemmatisation, and abbreviation expansion
class TextPreprocessor:

    #Initialise the text preprocessor
    def __init__(self, config: ProjectConfig):
        self.config = config

        #Initialise NLP components
        self.stop_words = set(stopwords.words('english')) if config.REMOVE_STOPWORDS else set()
        self.lemmatizer = WordNetLemmatizer() #For reducing words to base form
        self.stemmer = PorterStemmer() if config.USE_STEMMING else None #Optional stemming

        #Common electronics abbreviations and their full forms
        self.abbreviation_map = {
            #Storage units
            'gb': 'gigabyte', 'mb': 'megabyte', 'tb': 'terabyte', 'kb': 'kilobyte',
            #Processor speeds
            'ghz': 'gigahertz', 'mhz': 'megahertz', 'hz': 'hertz',
            #Memory and storage
            'ram': 'random access memory', 'rom': 'read only memory',
            'ssd': 'solid state drive', 'hdd': 'hard disk drive',
            'nvme': 'non-volatile memory express',
            #Computer components
            'cpu': 'central processing unit', 'gpu': 'graphics processing unit',
            'apu': 'accelerated processing unit',
            #Connectivity
            'wifi': 'wi-fi', 'bluetooth': 'bluetooth', 'nfc': 'near field communication',
            'usb': 'universal serial bus', 'hdmi': 'high definition multimedia interface',
            'displayport': 'display port', 'vga': 'video graphics array',
            #Display technologies
            'oled': 'organic light emitting diode',
            'led': 'light emitting diode', 'lcd': 'liquid crystal display',
            'ips': 'in-plane switching', 'tn': 'twisted nematic',
            'va': 'vertical alignment', 'amoled': 'active matrix organic led',
            #Brands and common terms
            'intel': 'intel', 'amd': 'amd', 'nvidia': 'nvidia', 'apple': 'apple',
            'samsung': 'samsung', 'lenovo': 'lenovo', 'dell': 'dell', 'hp': 'hp',
            #Other common abbreviations
            'os': 'operating system', 'gbps': 'gigabits per second',
            'mbps': 'megabits per second', 'fps': 'frames per second',
            'rgb': 'red green blue', 'argb': 'addressable rgb'
        }

        #Common misspellings and variations
        self.spelling_corrections = {
            'color': 'colour', 'colors': 'colours', 'center': 'centre',
            'favorite': 'favourite', 'organize': 'organise',
            'realize': 'realise', 'recognize': 'recognise',
            'iphone': 'iphone', 'ipad': 'ipad', 'imac': 'imac',
            'macbook': 'macbook', 'airpod': 'airpod'
        }

    #Clean and normalise text by removing special characters, URLs, etc
    #Cleaned text string is returned
    def clean_text(self, text: str) -> str:
        if not isinstance(text, str) or pd.isna(text): #Handle None or NaN values
            return ""
        #Convert to lowercase
        text = text.lower().strip()
        #Remove URLs
        text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
        #Remove HTML tags
        text = re.sub(r'<.*?>', '', text)
        #Remove special characters but keep alphanumeric, spaces, hyphens, and apostrophes
        text = re.sub(r'[^\w\s\-\']', ' ', text)
        #Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    #Expand common electronics abbreviations to full form
    #Text with abbreviations expanded is returned
    def expand_abbreviations(self, text: str) -> str:
        words = text.split() #Split text into individual words
        expanded_words = [] #Store expanded words

        for word in words:
             #Check for abbreviations with numbers (e.g., "8gb", "256gb")
            match = re.match(r'(\d+)([a-z]+)', word) #Pattern of number followed by letters
            if match:
                number, unit = match.groups() #Extract number and abbreviation
                if unit in self.abbreviation_map: #If the abbreviation is know
                    expanded_words.append(f"{number} {self.abbreviation_map[unit]}")
                    continue

            #Check for standalone abbreviations
            if word in self.abbreviation_map:
                expanded_words.append(self.abbreviation_map[word]) #Expand abbreviation
            else:
                expanded_words.append(word) #Keep word as is

        return ' '.join(expanded_words) #Join words back into text


    #Correct common spelling variations
    #Will return text with spelling corrections
    def correct_spelling(self, text: str) -> str:
        words = text.split()
        corrected_words = []

        for word in words:
            if word in self.spelling_corrections: #Check if words need any correction
                corrected_words.append(self.spelling_corrections[word]) #Apply correction
            else:
                corrected_words.append(word) #Keep original

        return ' '.join(corrected_words)

    def remove_stopwords(self, text: str) -> str:

        if not self.config.REMOVE_STOPWORDS: #Check if stopword removal is enabled
            return text

        words = text.split()
        filtered_words = [word for word in words if word not in self.stop_words] #Filter out stopwrods

        return ' '.join(filtered_words)

    #Apply lemmatisation to reduce words to their base form
    def lemmatize_text(self, text: str) -> str:
        words = text.split()
        lemmatized_words = [self.lemmatizer.lemmatize(word) for word in words] #Apply lemmatisation

        return ' '.join(lemmatized_words)

    #Apply stemming to reduce words to their root form
    def stem_text(self, text: str) -> str:
        if not self.config.USE_STEMMING: #Check if stemming is enabled
            return text

        words = text.split()
        stemmed_words = [self.stemmer.stem(word) for word in words] #Apply stemming

        return ' '.join(stemmed_words)


      #Preprocess all text columns in product data
    #Returns DataFrame with cleaned and processed text columns
    def preprocess_product_text(self, product_data: pd.DataFrame) -> pd.DataFrame:
        print(" Preprocessing text data...")

        #Identify text columns
        text_columns = [
            'title', 'shelfdescr', 'shortdescr', 'longdescr',
            'orig_shelfdescr', 'orig_shortdescr', 'orig_longdescr'
        ]

        #Filter to columns that exist in the data
        existing_text_cols = [col for col in text_columns if col in product_data.columns]

        if not existing_text_cols:
            print(" No text columns found for preprocessing")
            return product_data

        #Create processed versions of each text column
        for col in tqdm(existing_text_cols, desc="Processing text columns"):
            clean_col = f'clean_{col}' #New column name for cleaned text

            #Apply preprocessing pipeline
            product_data[clean_col] = product_data[col].apply(self.clean_text)
            product_data[clean_col] = product_data[clean_col].apply(self.expand_abbreviations)
            product_data[clean_col] = product_data[clean_col].apply(self.correct_spelling)
            product_data[clean_col] = product_data[clean_col].apply(self.remove_stopwords)
            product_data[clean_col] = product_data[clean_col].apply(self.lemmatize_text)

            #Truncate if too long
            product_data[clean_col] = product_data[clean_col].apply(
                lambda x: x[:self.config.MAX_TEXT_LENGTH] if len(x) > self.config.MAX_TEXT_LENGTH else x
            )

        #Create combined text column for embedding generation
        print(" Creating combined text column...")
        clean_cols = [f'clean_{col}' for col in existing_text_cols]

        #Combine all clean text columns
        combined_text_parts = []
        for col in clean_cols:
            if col in product_data.columns:
                combined_text_parts.append(product_data[col].fillna('')) #Handle missing values

        if combined_text_parts:
            #Start with the first column
            product_data['combined_text'] = combined_text_parts[0]

            #Add other columns with space separation
            for part in combined_text_parts[1:]:
                product_data['combined_text'] = product_data['combined_text'] + ' ' + part

            #Clean up extra spaces
            product_data['combined_text'] = product_data['combined_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

        print(f"Text preprocessing complete! Created {len(existing_text_cols)} clean columns.")

        return product_data

#Initialise text preprocessor
text_preprocessor = TextPreprocessor(config)
print("Text preprocessor initialised successfully!")

Feature engineer for random forest

In [ ]:
#Creates features for the Random Forest classifier
#Extracts text similarity, categorical matching, and numerical features
class FeatureEngineer:

    #Initialise the feature engineer
    def __init__(self, config: ProjectConfig):

        self.config = config
        #TF-IDF vectorizer for text similarity
        self.tfidf_vectorizer = None
        self.tfidf_fitted = False

        #Common electronics abbreviations mapping
        self.abbreviation_map = {
            'gb': 'gigabyte', 'mb': 'megabyte', 'tb': 'terabyte', 'kb': 'kilobyte',
            'ghz': 'gigahertz', 'mhz': 'megahertz', 'hz': 'hertz',
            'ram': 'random access memory', 'rom': 'read only memory',
            'ssd': 'solid state drive', 'hdd': 'hard disk drive',
            'nvme': 'non-volatile memory express',
            'cpu': 'central processing unit', 'gpu': 'graphics processing unit',
            'apu': 'accelerated processing unit',
            'wifi': 'wi-fi', 'bluetooth': 'bluetooth', 'nfc': 'near field communication',
            'usb': 'universal serial bus', 'hdmi': 'high definition multimedia interface',
            'displayport': 'display port', 'vga': 'video graphics array',
            'oled': 'organic light emitting diode',
            'led': 'light emitting diode', 'lcd': 'liquid crystal display',
            'ips': 'in-plane switching', 'tn': 'twisted nematic',
            'va': 'vertical alignment', 'amoled': 'active matrix organic led',
            'intel': 'intel', 'amd': 'amd', 'nvidia': 'nvidia', 'apple': 'apple',
            'samsung': 'samsung', 'lenovo': 'lenovo', 'dell': 'dell', 'hp': 'hp',
            'os': 'operating system', 'gbps': 'gigabits per second',
            'mbps': 'megabits per second', 'fps': 'frames per second',
            'rgb': 'red green blue', 'argb': 'addressable rgb'
        }

    #Fit TF-IDF on training corpus only
    def fit_tfidf(self, corpus_texts: List[str]):

        print("Fitting TF-IDF vectorizer on training corpus...")

        if not corpus_texts or len(corpus_texts) == 0:
            corpus_texts = ["electronics product computer laptop"]

        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=100,
            stop_words='english',
            ngram_range=(1, 2)
        )
        self.tfidf_vectorizer.fit(corpus_texts)
        self.tfidf_fitted = True
        print(f"  TF-IDF vocabulary size: {len(self.tfidf_vectorizer.vocabulary_)}")

    #Compute multiple text similarity measures between two strings
    def compute_text_similarity_features(self, text1: str, text2: str) -> Dict[str, float]:

        if not text1 or not text2:
            return {
                'levenshtein_similarity': 0.0,
                'jaccard_similarity': 0.0,
                'cosine_similarity': 0.0,
                'dice_coefficient': 0.0
            }

        #Levenshtein similarity
        lev_distance = Levenshtein.distance(text1, text2)
        max_len = max(len(text1), len(text2))
        levenshtein_similarity = 1 - (lev_distance / max_len) if max_len > 0 else 0

        #Jaccard similarity
        tokens1 = set(text1.lower().split())
        tokens2 = set(text2.lower().split())

        intersection = len(tokens1.intersection(tokens2))
        union = len(tokens1.union(tokens2))
        jaccard_similarity = intersection / union if union > 0 else 0

        #Cosine similarity only if TF-IDF is fitted
        cosine_sim = 0.0
        if self.tfidf_fitted and self.tfidf_vectorizer is not None:
            try:
                tfidf_matrix = self.tfidf_vectorizer.transform([text1, text2])
                from sklearn.metrics.pairwise import cosine_similarity
                cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            except Exception as e:
                cosine_sim = 0.0
        else:
            cosine_sim = 0.0  #Will be 0 until TF-IDF is fitted

        #Dice coefficient
        dice_coefficient = (2 * intersection) / (len(tokens1) + len(tokens2)) if (len(tokens1) + len(tokens2)) > 0 else 0

        return {
            'levenshtein_similarity': round(levenshtein_similarity, 4),
            'jaccard_similarity': round(jaccard_similarity, 4),
            'cosine_similarity': round(cosine_sim, 4),
            'dice_coefficient': round(dice_coefficient, 4)
        }

    #Compute features for categorical attributes
    def compute_categorical_features(self, value1: Any, value2: Any) -> Dict[str, float]:

        str1 = str(value1).lower().strip() if not pd.isna(value1) else ""
        str2 = str(value2).lower().strip() if not pd.isna(value2) else ""

        if not str1 or not str2:
            return {
                'levenshtein_ratio': 0.0,
                'token_overlap': 0.0,
                'length_similarity': 0.0
            }

        #Levenshtein ratio
        levenshtein_ratio = Levenshtein.ratio(str1, str2)

        #Token overlap
        tokens1 = set(str1.split())
        tokens2 = set(str2.split())
        token_overlap = len(tokens1.intersection(tokens2)) / max(len(tokens1), len(tokens2)) if max(len(tokens1), len(tokens2)) > 0 else 0.0

        #Length similarity
        len_similarity = min(len(str1), len(str2)) / max(len(str1), len(str2)) if max(len(str1), len(str2)) > 0 else 0.0

        return {
            'levenshtein_ratio': round(levenshtein_ratio, 4),
            'token_overlap': round(token_overlap, 4),
            'length_similarity': round(len_similarity, 4)
        }

    #Compute brand features
    def compute_brand_features(self, brand1: str, brand2: str) -> Dict[str, float]:

        brand_features = self.compute_categorical_features(brand1, brand2)

        has_brand = 1.0 if (brand1 and brand2) else 0.0
        brand_features['has_brand'] = has_brand

        return brand_features

    #Compute price features
    def compute_price_features(self, price1: Any, price2: Any) -> Dict[str, float]:
    #Handles the messy price data which has dollar signs
        try:
            if isinstance(price1, str):
                price1 = price1.replace('$', '').strip()
                if price1.lower() == 'nan' or price1 == '':
                    p1 = 0.0
                else:
                    p1 = float(price1)
            else:
                p1 = float(price1) if not pd.isna(price1) else 0.0

            if isinstance(price2, str):
                price2 = price2.replace('$', '').strip()
                if price2.lower() == 'nan' or price2 == '':
                    p2 = 0.0
                else:
                    p2 = float(price2)
            else:
                p2 = float(price2) if not pd.isna(price2) else 0.0

            if p1 == 0 or p2 == 0:
                return {
                    'price_ratio': 0.0,
                    'price_log_similarity': 0.0,
                    'has_price': 0.0
                }

            price_ratio = min(p1, p2) / max(p1, p2) if max(p1, p2) > 0 else 0.0

            price_log_ratio = abs(np.log(p1) - np.log(p2)) if p1 > 0 and p2 > 0 else 10.0
            price_log_similarity = 1.0 / (1.0 + price_log_ratio) if price_log_ratio > 0 else 0.0

            return {
                'price_ratio': round(price_ratio, 4),
                'price_log_similarity': round(price_log_similarity, 4),
                'has_price': 1.0
            }

        except (ValueError, TypeError, AttributeError):
            return {
                'price_ratio': 0.0,
                'price_log_similarity': 0.0,
                'has_price': 0.0
            }

    #Create features for a product pair
    def create_features_for_pair(self, product_a: pd.Series, product_b: pd.Series) -> Dict[str, Any]:
        features = {}

        #Text similarity features
        if self.config.USE_TEXT_SIMILARITY:
            text_a = self._get_text_without_identifiers(product_a)
            text_b = self._get_text_without_identifiers(product_b)

            text_features = self.compute_text_similarity_features(text_a, text_b)
            features.update({f'text_{k}': v for k, v in text_features.items()})

            words_a = set(text_a.lower().split())
            words_b = set(text_b.lower().split())
            word_overlap = len(words_a.intersection(words_b)) / max(len(words_a), len(words_b)) if max(len(words_a), len(words_b)) > 0 else 0.0
            features['word_overlap'] = round(word_overlap, 4)

        #Categorical features
        if self.config.USE_CATEGORICAL_FEATURES:
            brand1 = str(product_a.get('brand', '')).lower().strip() if not pd.isna(product_a.get('brand')) else ""
            brand2 = str(product_b.get('brand', '')).lower().strip() if not pd.isna(product_b.get('brand')) else ""

            if brand1 and brand2:
                brand_features = self.compute_brand_features(brand1, brand2)
                features.update({f'brand_{k}': v for k, v in brand_features.items()})
            else:
                features.update({
                    'brand_levenshtein_ratio': 0.0,
                    'brand_token_overlap': 0.0,
                    'brand_length_similarity': 0.0,
                    'brand_has_brand': 0.0
                })

            group1 = str(product_a.get('groupname', '')).lower().strip() if not pd.isna(product_a.get('groupname')) else ""
            group2 = str(product_b.get('groupname', '')).lower().strip() if not pd.isna(product_b.get('groupname')) else ""

            if group1 and group2:
                group_features = self.compute_categorical_features(group1, group2)
                features.update({f'group_{k}': v for k, v in group_features.items()})
            else:
                features.update({
                    'group_levenshtein_ratio': 0.0,
                    'group_token_overlap': 0.0,
                    'group_length_similarity': 0.0
                })

        #Numerical features
        if self.config.USE_NUMERICAL_FEATURES:
            price1 = product_a.get('price', 0)
            price2 = product_b.get('price', 0)
            price_features = self.compute_price_features(price1, price2)
            features.update(price_features)

            if 'shipweight' in product_a and 'shipweight' in product_b:
                weight1 = product_a.get('shipweight', 0)
                weight2 = product_b.get('shipweight', 0)
                try:
                    w1 = float(weight1) if not pd.isna(weight1) else 0.0
                    w2 = float(weight2) if not pd.isna(weight2) else 0.0
                    if w1 > 0 and w2 > 0:
                        weight_ratio = min(w1, w2) / max(w1, w2)
                        features['weight_ratio'] = round(weight_ratio, 4)
                    else:
                        features['weight_ratio'] = 0.0
                except:
                    features['weight_ratio'] = 0.0

        features['has_brand_both'] = 1.0 if (brand1 and brand2) else 0.0
        features['has_group_both'] = 1.0 if (group1 and group2) else 0.0

        len_a = len(str(text_a))
        len_b = len(str(text_b))
        features['text_len_diff'] = abs(len_a - len_b) / max(len_a, len_b) if max(len_a, len_b) > 0 else 0.0

        validated_features = {}
        for key, value in features.items():
            try:
                float_value = float(value)
                if np.isinf(float_value):
                    float_value = 1.0 if float_value > 0 else 0.0
                if np.isnan(float_value):
                    float_value = 0.0
                validated_features[key] = round(float_value, 4)
            except (ValueError, TypeError):
                validated_features[key] = value

        return validated_features

    #Get text without identifiers
    def _get_text_without_identifiers(self, product: pd.Series) -> str:

        text_parts = []

        title = str(product.get('clean_title', product.get('title', '')))
        title = re.sub(r'\\b[A-Z0-9]{6,}\\b', '', title)
        title = re.sub(r'\\b\\d+\\s*(gb|mb|tb|ghz|mhz|hz|inches|inch)\\b', '', title, flags=re.IGNORECASE)
        text_parts.append(title.strip())

        for desc_field in ['clean_shortdescr', 'clean_longdescr', 'clean_shelfdescr']:
            desc = str(product.get(desc_field, ''))
            if desc and desc.lower() != 'nan':
                desc = re.sub(r'\\b[A-Z0-9]{6,}\\b', '', desc)
                text_parts.append(desc.strip())

        combined = ' '.join(text_parts)
        combined = re.sub(r'\\s+', ' ', combined).strip()

        return combined

#Initialise feature engineer
feature_engineer = FeatureEngineer(config)
print("Feature engineer initialised successfully!")

Dataset Creator class

In [ ]:
#Creates labelled product pairs for training and evaluation and handles positive pair generation and negative sampling
class DatasetCreator:

    #Initialise the dataset creator
    def __init__(self, config: ProjectConfig):

        self.config = config
        self.text_preprocessor = TextPreprocessor(config)
        self.feature_engineer = FeatureEngineer(config)

    #Load verified matches from matches.csv file
    #This file contains ground truth matching pairs with id1 from tableA and id2 from tableB
    #id1 and id2 correspond to the custom_id column in each table, not row indices
    #Returns the matches DataFrame
    def load_ground_truth_matches(self) -> pd.DataFrame:

        print("Loading ground truth matches from matches.csv...")

        #Construct file path for matches.csv
        matches_path = os.path.join(self.config.DATA_PATH, 'matches.csv')

        #Check if file exists
        if not os.path.exists(matches_path):
            raise FileNotFoundError(
                f"matches.csv not found at: {matches_path}\n"
                f"Please ensure matches.csv is in your data folder."
            )

        #Load the matches file
        matches_df = pd.read_csv(matches_path)

        #Verify the file has the expected columns
        if 'id1' not in matches_df.columns or 'id2' not in matches_df.columns:
            raise ValueError("matches.csv must contain 'id1' and 'id2' columns")

        print(f"Successfully loaded {len(matches_df)} verified matches")
        print(f"Columns: {list(matches_df.columns)}")

        return matches_df

    #Create positive pairs from ground truth matches loaded from matches.csv
    #id1 and id2 are custom_id values (1-indexed) so they are converted to 0-based row indices
    #Returns list of tuples containing row indices from tableA and tableB that are verified matches
    def create_positive_pairs_from_matches(self, table_a: pd.DataFrame,
                                          table_b: pd.DataFrame) -> List[Tuple[int, int]]:

        print("\nCreating positive pairs from verified matches...")

        #Load verified matches from matches.csv
        matches_df = self.load_ground_truth_matches()

        #Build a mapping from custom_id to row index for both tables
        #custom_id starts at 1 so custom_id - 1 gives the correct row index
        custom_id_to_row_a = {int(cid): idx for idx, cid in enumerate(table_a['custom_id'])}
        custom_id_to_row_b = {int(cid): idx for idx, cid in enumerate(table_b['custom_id'])}

        #Convert matches to list of tuples using row indices
        positive_pairs = []
        skipped = 0

        for _, row in matches_df.iterrows():
            id1 = int(row['id1'])
            id2 = int(row['id2'])

            #Convert custom_id to row index using the mapping
            row_a = custom_id_to_row_a.get(id1)
            row_b = custom_id_to_row_b.get(id2)

            #Verify both custom_ids exist in the tables
            if row_a is not None and row_b is not None:
                positive_pairs.append((row_a, row_b))
            else:
                skipped += 1

        if skipped > 0:
            print(f"Warning: Skipped {skipped} pairs where custom_id was not found in tables")

        print(f"Created {len(positive_pairs)} positive pairs from verified matches")

        return positive_pairs

    #Create negative pairs using hard negative mining strategy
    #Hard negatives are products that are similar but not matches (e.g., same brand, same category, similar price)
    #This creates a realistic and challenging product matching task
    #Returns list of tuples containing non-matching product row indices
    def create_negative_pairs(self, table_a: pd.DataFrame, table_b: pd.DataFrame,
                             positive_pairs: List[Tuple[int, int]],
                             n_negative: int) -> List[Tuple[int, int]]:

        print(f"Creating {n_negative} hard negative pairs...")
        print("Hard negative strategy: same brand OR similar price OR random")

        negative_pairs = []
        max_attempts = n_negative * 20
        positive_set = set(positive_pairs)

        #Helper to clean price strings
        def clean_price(price_val):
            try:
                if pd.isna(price_val) or price_val == 0:
                    return None
                if isinstance(price_val, str):
                    price_val = price_val.replace('$', '').strip()
                    if price_val.lower() == 'nan' or price_val == '':
                        return None
                return float(price_val)
            except:
                return None

        #Group products by brand
        brand_to_products_a = {}
        brand_to_products_b = {}

        for idx, row in table_a.iterrows():
            brand = str(row.get('brand', '')).lower().strip()
            if brand and brand != 'nan' and brand != '':
                if brand not in brand_to_products_a:
                    brand_to_products_a[brand] = []
                brand_to_products_a[brand].append(idx)

        for idx, row in table_b.iterrows():
            brand = str(row.get('brand', '')).lower().strip()
            if brand and brand != 'nan' and brand != '':
                if brand not in brand_to_products_b:
                    brand_to_products_b[brand] = []
                brand_to_products_b[brand].append(idx)

        #Group products by price range
        def get_price_bucket(price):
            p = clean_price(price)
            if p is None:
                return None
            if p < 20:
                return 'under_20'
            elif p < 50:
                return '20_to_50'
            elif p < 100:
                return '50_to_100'
            elif p < 200:
                return '100_to_200'
            elif p < 500:
                return '200_to_500'
            else:
                return 'over_500'

        price_to_products_a = {}
        price_to_products_b = {}

        for idx, row in table_a.iterrows():
            bucket = get_price_bucket(row.get('price'))
            if bucket:
                if bucket not in price_to_products_a:
                    price_to_products_a[bucket] = []
                price_to_products_a[bucket].append(idx)

        for idx, row in table_b.iterrows():
            bucket = get_price_bucket(row.get('price'))
            if bucket:
                if bucket not in price_to_products_b:
                    price_to_products_b[bucket] = []
                price_to_products_b[bucket].append(idx)

        #Progress bar for negative pair generation
        with tqdm(total=n_negative, desc="Generating hard negative pairs") as pbar:
            attempts = 0
            same_brand_count = 0
            same_price_count = 0
            random_count = 0

            while len(negative_pairs) < n_negative and attempts < max_attempts:
                attempts += 1

                #Strategy: 40% same brand, 30% same price range, 30% random
                strategy = random.random()

                if strategy < 0.4:
                    #Try to get same brand pair
                    common_brands = set(brand_to_products_a.keys()) & set(brand_to_products_b.keys())
                    if common_brands:
                        brand = random.choice(list(common_brands))
                        idx_a = random.choice(brand_to_products_a[brand])
                        idx_b = random.choice(brand_to_products_b[brand])

                        if (idx_a, idx_b) not in positive_set and (idx_a, idx_b) not in negative_pairs:
                            negative_pairs.append((idx_a, idx_b))
                            same_brand_count += 1
                            pbar.update(1)
                        continue

                elif strategy < 0.7:
                    #Try to get same price range pair
                    common_buckets = set(price_to_products_a.keys()) & set(price_to_products_b.keys())
                    if common_buckets:
                        bucket = random.choice(list(common_buckets))
                        idx_a = random.choice(price_to_products_a[bucket])
                        idx_b = random.choice(price_to_products_b[bucket])

                        if (idx_a, idx_b) not in positive_set and (idx_a, idx_b) not in negative_pairs:
                            negative_pairs.append((idx_a, idx_b))
                            same_price_count += 1
                            pbar.update(1)
                        continue

                #Random sampling (fallback or 30% of the time)
                idx_a = random.randint(0, len(table_a) - 1)
                idx_b = random.randint(0, len(table_b) - 1)

                if (idx_a, idx_b) not in positive_set and (idx_a, idx_b) not in negative_pairs:
                    negative_pairs.append((idx_a, idx_b))
                    random_count += 1
                    pbar.update(1)

        print(f"Created {len(negative_pairs)} negative pairs:")
        print(f"  - Same brand: {same_brand_count} ({same_brand_count/len(negative_pairs)*100:.1f}%)")
        print(f"  - Same price range: {same_price_count} ({same_price_count/len(negative_pairs)*100:.1f}%)")
        print(f"  - Random: {random_count} ({random_count/len(negative_pairs)*100:.1f}%)")

        if len(negative_pairs) < n_negative:
            print(f"Warning: Only generated {len(negative_pairs)} negative pairs (target: {n_negative})")

        return negative_pairs

    #Create labelled dataset with train, validation, and test splits
    #Uses verified matches from matches.csv for positive pairs and hard negative mining for negative pairs
    #Returns pairs DataFrame, cleaned tables, and splits dictionary
    def create_labelled_dataset(self, table_a: pd.DataFrame, table_b: pd.DataFrame,
                               positive_pairs: Optional[List[Tuple[int, int]]] = None) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict]:

        print("Creating labelled dataset...")

        #Preprocess text for both tables
        print("Step 1: Preprocessing text...")
        table_a_clean = self.text_preprocessor.preprocess_product_text(table_a.copy())
        table_b_clean = self.text_preprocessor.preprocess_product_text(table_b.copy())

        #Create positive pairs from ground truth matches in matches.csv
        if positive_pairs is None:
            positive_pairs = self.create_positive_pairs_from_matches(table_a_clean, table_b_clean)

        n_positive = len(positive_pairs)
        n_negative = int(n_positive * self.config.NEGATIVE_TO_POSITIVE_RATIO)

        #Create hard negative pairs
        negative_pairs = self.create_negative_pairs(table_a_clean, table_b_clean, positive_pairs, n_negative)

        #Create all pairs without features first (just indices and labels)
        print("Step 2: Creating pair indices and labels...")
        all_pairs_indices = []

        #Add positive pairs with label 1
        for idx_a, idx_b in positive_pairs:
            all_pairs_indices.append((idx_a, idx_b, 1))

        #Add negative pairs with label 0
        for idx_a, idx_b in negative_pairs:
            all_pairs_indices.append((idx_a, idx_b, 0))

        #Create initial DataFrame with just indices and labels
        pairs_indices_df = pd.DataFrame(all_pairs_indices, columns=['idx_a', 'idx_b', 'label'])

        #Shuffle the dataset
        pairs_indices_df = pairs_indices_df.sample(frac=1, random_state=self.config.RANDOM_STATE).reset_index(drop=True)

        #Split dataset before fitting TF-IDF to prevent data leakage
        print("Step 3: Splitting dataset before feature extraction...")
        splits = self.split_dataset(pairs_indices_df)

        #Fit TF-IDF only on training data to prevent information leakage
        print("Step 4: Fitting TF-IDF ONLY on training data...")
        train_df = splits['train']
        training_texts = []

        #Collect text from training pairs only for TF-IDF fitting
        for _, row in train_df.iterrows():
            idx_a = int(row['idx_a'])
            idx_b = int(row['idx_b'])

            if idx_a < len(table_a_clean) and idx_b < len(table_b_clean):
                product_a = table_a_clean.iloc[idx_a]
                product_b = table_b_clean.iloc[idx_b]

                text_a = self.feature_engineer._get_text_without_identifiers(product_a)
                text_b = self.feature_engineer._get_text_without_identifiers(product_b)

                if text_a:
                    training_texts.append(text_a)
                if text_b:
                    training_texts.append(text_b)

        #Fit TF-IDF on training corpus only
        self.feature_engineer.fit_tfidf(training_texts)

        #Extract features for each split using the fitted TF-IDF
        print(f"Step 5: Extracting features for each split...")

        #Process each split separately
        final_splits = {}
        for split_name, split_df in splits.items():
            print(f"  Processing {split_name} split ({len(split_df)} pairs)...")

            split_features = []
            for _, row in tqdm(split_df.iterrows(), desc=f"{split_name} pairs", total=len(split_df)):
                idx_a = int(row['idx_a'])
                idx_b = int(row['idx_b'])
                label = int(row['label'])

                if idx_a < len(table_a_clean) and idx_b < len(table_b_clean):
                    product_a = table_a_clean.iloc[idx_a]
                    product_b = table_b_clean.iloc[idx_b]

                    #Extract features using the feature engineer
                    features = self.feature_engineer.create_features_for_pair(product_a, product_b)

                    #Add label and indices
                    features['label'] = label
                    features['idx_a'] = idx_a
                    features['idx_b'] = idx_b

                    split_features.append(features)

            #Create DataFrame for this split
            final_splits[split_name] = pd.DataFrame(split_features)

        #Combine all splits for the full dataset
        print("Step 6: Creating combined dataset...")
        all_features_list = []

        for split_name in ['train', 'validation', 'test']:
            all_features_list.append(final_splits[split_name])

        pairs_df = pd.concat(all_features_list, ignore_index=True)

        #Print dataset summary
        print(f"\n DATASET SUMMARY:")
        print(f"Total pairs: {len(pairs_df)}")
        print(f"Positive matches: {len(pairs_df[pairs_df['label'] == 1])} ({len(pairs_df[pairs_df['label'] == 1])/len(pairs_df)*100:.1f}%)")
        print(f"Negative non-matches: {len(pairs_df[pairs_df['label'] == 0])} ({len(pairs_df[pairs_df['label'] == 0])/len(pairs_df)*100:.1f}%)")
        print(f"Number of features: {len([c for c in pairs_df.columns if c not in ['label', 'idx_a', 'idx_b']])}")
        print(f"TF-IDF fitted only on: {len(train_df)} training samples")

        #Save splits to CSV
        train_save_path = os.path.join(self.config.DATA_PATH, 'train_split.csv')
        val_save_path = os.path.join(self.config.DATA_PATH, 'validation_split.csv')
        test_save_path = os.path.join(self.config.DATA_PATH, 'test_split.csv')

        final_splits['train'].to_csv(train_save_path, index=False)
        final_splits['validation'].to_csv(val_save_path, index=False)
        final_splits['test'].to_csv(test_save_path, index=False)

        print(f"\n Dataset preparation complete!")
        print(f"   Training features shape: {final_splits['train'].shape}")
        print(f"   Testing features shape: {final_splits['test'].shape}")

        #Return all four components (pairs_df, cleaned tables, and splits)
        return pairs_df, table_a_clean, table_b_clean, final_splits

    #Split the dataset into training, validation, and test sets
    #Stratified split maintains class balance across all splits
    def split_dataset(self, pairs_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:

        print(f"\n Splitting dataset...")

        #First split into train+val and test
        train_val, test = train_test_split(
            pairs_df,
            test_size=self.config.TEST_SIZE,
            stratify=pairs_df['label'],
            random_state=self.config.RANDOM_STATE
        )

        #Calculate validation size relative to train_val
        val_size_relative = self.config.VALIDATION_SIZE / (1 - self.config.TEST_SIZE)

        #Split train_val into train and validation
        train, validation = train_test_split(
            train_val,
            test_size=val_size_relative,
            stratify=train_val['label'],
            random_state=self.config.RANDOM_STATE
        )

        print("Dataset split complete:")
        print(f"  Training set: {len(train)} samples ({len(train)/len(pairs_df)*100:.1f}%)")
        print(f"  Validation set: {len(validation)} samples ({len(validation)/len(pairs_df)*100:.1f}%)")
        print(f"  Test set: {len(test)} samples ({len(test)/len(pairs_df)*100:.1f}%)")

        return {
            'train': train.reset_index(drop=True),
            'validation': validation.reset_index(drop=True),
            'test': test.reset_index(drop=True)
        }

#Initialise dataset creator
dataset_creator = DatasetCreator(config)
print("Dataset creator initialised successfully!")

Random forest model class

In [ ]:
#Random Forest classifier for product matching
#Implements traditional machine learning approach with feature engineering
class RandomForestProductMatcher:
    #Initialise the Random Forest model.
    def __init__(self, config: ProjectConfig):

        self.config = config
        self.model = None #Will hold the trained RandomForestClassifier
        self.is_trained = False #Flag to track training status
        self.feature_importances = None #Will store feature importance scores
        self.scaler = StandardScaler() #For feature standarisation
        self.best_params = None #Store best hyperparameters found during tuning

        #Initialise model with configured parameters
        self._initialise_model()

    def _initialise_model(self):
        #Initialise the Random Forest classifier with configured parameters
        self.model = RandomForestClassifier(
            n_estimators=self.config.RF_N_ESTIMATORS, #Number of decision trees
            max_depth=self.config.RF_MAX_DEPTH, #Maximum tree depth
            min_samples_split=self.config.RF_MIN_SAMPLES_SPLIT, #Minimum samples to split node
            min_samples_leaf=self.config.RF_MIN_SAMPLES_LEAF, #Minimum samples at leaf node
            random_state=self.config.RANDOM_STATE, #For reproducibility
            n_jobs=-1,  # Use all available CPU cores
            class_weight='balanced',  # Handle class imbalance
            bootstrap=True, #Use bootstrap sampling
            oob_score=True, #Calculate the out of bag score
            verbose=0 #No training verbosity
        )


    #Prepare features by handling missing values and scaling
    def prepare_features(self, X: pd.DataFrame, fit_scaler: bool = False) -> np.ndarray:
        #Handle missing values by filling with 0
        X_filled = X.fillna(0)

        #Convert to numpy array
        X_array = X_filled.values

        #Scale features
        if fit_scaler:
            X_scaled = self.scaler.fit_transform(X_array) #Fit and transform (training)
        else:
            X_scaled = self.scaler.transform(X_array) #Transform only (testing)

        return X_scaled

    #Train the Random Forest model.
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame = None, y_val: pd.Series = None) -> Dict[str, Any]:

        print("Training Random Forest model")
        start_time = time.time()

        #Hyperparameter tuning if enabled
        if self.config.TUNE_RANDOM_FOREST:
            print("Performing hyperparameter tuning")
            tuning_results = self.tune_hyperparameters(X_train, y_train)
            self.best_params = tuning_results['best_params']
            print(f"Best hyperparameters: {self.best_params}")

            #Reinitialise model with best parameters
            self.model.set_params(**self.best_params)

        #Prepare features
        print("Preparing features...")
        X_train_prepared = self.prepare_features(X_train, fit_scaler=True)

        #Train the model
        print("Fitting model")
        self.model.fit(X_train_prepared, y_train)
        self.is_trained = True

        #Calculate feature importances
        self.feature_importances = pd.DataFrame({
            'feature': X_train.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)

        #Calculate training metrics
        train_predictions = self.model.predict(X_train_prepared)
        train_metrics = self._calculate_metrics(y_train, train_predictions)

        training_time = time.time() - start_time

        #Validation metrics (if validation data provided)
        val_metrics = None
        if X_val is not None and y_val is not None:
            print("Evaluating on validation set...")
            X_val_prepared = self.prepare_features(X_val, fit_scaler=False)
            val_predictions = self.model.predict(X_val_prepared)
            val_metrics = self._calculate_metrics(y_val, val_predictions)

        #Print summary
        print(f"\n Random Forest training complete!")
        print(f"Training time: {training_time:.2f} seconds")
        print(f"Number of trees: {self.model.n_estimators}")
        print(f"Max depth: {self.model.max_depth}")
        print(f"Out-of-bag score: {self.model.oob_score_:.4f}")

        if self.best_params:
            print(f"Tuned hyperparameters: {self.best_params}")

        if val_metrics:
            print(f"Validation accuracy: {val_metrics['accuracy']:.4f}")
            print(f"Validation F1-score: {val_metrics['f1']:.4f}")

        #Return results
        results = {
            'training_time': training_time,
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'feature_importances': self.feature_importances,
            'oob_score': self.model.oob_score_,
            'best_params': self.best_params
        }

        return results

    #Make predictions with the trained model
    def predict(self, X: pd.DataFrame, return_proba: bool = False) -> np.ndarray:

        if not self.is_trained:
            raise ValueError("Model must be trained before making predictions")

        #Prepare features
        X_prepared = self.prepare_features(X, fit_scaler=False)

        if return_proba:
            return self.model.predict_proba(X_prepared)[:, 1]  #Probability of class 1 (a match)
        else:
            return self.model.predict(X_prepared) #Class labels (0 or 1)



    #Evaluate model performance on test data
    def evaluate(self, X_test: pd.DataFrame, y_test: pd.Series,
                 threshold: float = 0.5) -> Dict[str, Any]:

        print("\n Evaluating Random Forest model on test set...")

        #Get predictions and probabilities
        y_pred_proba = self.predict(X_test, return_proba=True)
        y_pred = (y_pred_proba >= threshold).astype(int) #Apply threshold

        #Calculate metrics
        metrics = self._calculate_metrics(y_test, y_pred, y_pred_proba)

        #Print results
        print(f"\n TEST SET PERFORMANCE:")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1-Score:  {metrics['f1']:.4f}")
        print(f"AUC-ROC:   {metrics['roc_auc']:.4f}")

        #Classification report
        print("\n CLASSIFICATION REPORT:")
        print(classification_report(y_test, y_pred, target_names=['Non-Match', 'Match']))

        return metrics


    #Calculate comprehensive evaluation metrics
    def _calculate_metrics(self, y_true: pd.Series, y_pred: np.ndarray,
                          y_pred_proba: Optional[np.ndarray] = None) -> Dict[str, float]:

        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }

        #Calculate AUC-ROC if probabilities are provided
        if y_pred_proba is not None:
            metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)

        return metrics


    #Perform cross-validation to detect overfitting.
    def cross_validate(self, X: pd.DataFrame, y: pd.Series, cv: int = 5) -> Dict[str, Any]:

        print("Performing cross-validation...")

        #Prepare features
        X_prepared = self.prepare_features(X, fit_scaler=True)

        #Create stratified k-fold for balanced class distribution in each fold
        from sklearn.model_selection import cross_val_score, StratifiedKFold

        cv_strategy = StratifiedKFold(n_splits=cv, shuffle=True, random_state=self.config.RANDOM_STATE)

        #Define the metrics to calculate during cross-validation
        metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        cv_results = {}

        for metric in metrics:
            if metric == 'roc_auc':
                scoring = 'roc_auc'
            elif metric == 'precision':
                scoring = 'precision'
            elif metric == 'recall':
                scoring = 'recall'
            elif metric == 'f1':
                scoring = 'f1'
            else:
                scoring = 'accuracy'

            #Calculate cross-validation scores for this metric
            scores = cross_val_score(
                self.model, X_prepared, y,
                cv=cv_strategy, scoring=scoring, n_jobs=-1
            )

            #Store detailed statistics
            cv_results[metric] = {
                'scores': scores.tolist(),
                'mean': float(scores.mean()),
                'std': float(scores.std()),
                'min': float(scores.min()),
                'max': float(scores.max())
            }
            #Print summary for this metric
            print(f"  {metric.upper()}: {scores.mean():.4f} ± {scores.std():.4f} (range: {scores.min():.4f}-{scores.max():.4f})")

        #Check for overfitting) if train-test gap is large
        if 'mean' in cv_results['accuracy']:
            train_score = self.model.score(X_prepared, y) #Score on entire training set
            cv_score = cv_results['accuracy']['mean']
            gap = train_score - cv_score #Training-CV gap indicates overfitting

            print(f"\n Overfitting Analysis:")
            print(f"   Training score: {train_score:.4f}")
            print(f"   CV score: {cv_score:.4f}")
            print(f"   Gap: {gap:.4f}")

            if gap > 0.05:
                print(f" WARNING: Potential overfitting (gap > 0.05)")
            elif gap > 0.02:
                print(f" Mild overfitting (gap > 0.02)")
            else:
                print(f" Good generalization")

        return cv_results


    #Tune hyperparameters using grid search
    def tune_hyperparameters(self, X_train: pd.DataFrame, y_train: pd.Series,
                            param_grid: Optional[Dict] = None, cv: int = None) -> Dict[str, Any]:

        print("Tuning Random Forest hyperparameters...")

        #Default parameter grid if not provided
        if param_grid is None:
            param_grid = {
                'n_estimators': [100, 200, 300, 400], #Number of trees to try
                'max_depth': [10, 20, 30, None], #Tree depth options
                'min_samples_split': [2, 5, 10], #Split the criteria
                'min_samples_leaf': [1, 2, 4], #The leaf criteria
                'max_features': ['sqrt', 'log2', None], #Feature sampling strategies
                'bootstrap': [True, False] #Whether bootstrap samples are used
            }

        #Use config CV folds if not specified
        if cv is None:
            cv = self.config.RF_TUNE_CV_FOLDS

        #Prepare features
        X_train_prepared = self.prepare_features(X_train, fit_scaler=True)

        #Perform grid search with cross-validation
        grid_search = GridSearchCV(
            estimator=RandomForestClassifier(
                random_state=self.config.RANDOM_STATE,
                class_weight='balanced',
                n_jobs=-1,
                oob_score=True
            ),
            param_grid=param_grid,
            cv=cv,
            scoring='f1', #Optimise the F1-Score
            n_jobs=self.config.RF_TUNE_N_JOBS,
            verbose=1,
            refit=True #Refit the best estimator on the entire training set
        )

        #Fit grid search
        print(f"Grid search with {len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['min_samples_split']) * len(param_grid['min_samples_leaf']) * len(param_grid['max_features']) * len(param_grid['bootstrap'])} parameter combinations...")
        grid_search.fit(X_train_prepared, y_train)

        #Update model with best parameters
        self.model = grid_search.best_estimator_
        self.is_trained = True
        self.best_params = grid_search.best_params_

        #Print results
        print(f"\n Hyperparameter tuning complete!")
        print(f"Best parameters: {grid_search.best_params_}")
        print(f"Best F1-score: {grid_search.best_score_:.4f}")

        return {
            'best_params': grid_search.best_params_,
            'best_score': grid_search.best_score_,
            'cv_results': grid_search.cv_results_
        }

    #Save trained model
    def save_model(self, filepath: str):

        if not self.is_trained:
            raise ValueError("Model must be trained before saving")

        import joblib

        #Create directory if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)

        #Save model and scaler
        model_data = {
            'model': self.model,
            'scaler': self.scaler,
            'feature_importances': self.feature_importances,
            'config': self.config,
            'best_params': self.best_params
        }

        joblib.dump(model_data, filepath)
        print(f" Model saved to: {filepath}")

    #Load trained model
    def load_model(self, filepath: str):

        import joblib

        #Load model data
        model_data = joblib.load(filepath)

        #Restore model components
        self.model = model_data['model']
        self.scaler = model_data['scaler']
        self.feature_importances = model_data['feature_importances']
        self.best_params = model_data.get('best_params', None)
        self.is_trained = True

        print(f"Model loaded from: {filepath}")

#Initialise Random Forest model
rf_model = RandomForestProductMatcher(config)
print("Random Forest model initialised successfully!")

Siamese Network Model class

In [ ]:
#PyTorch Dataset for Siamese Network training
#Provides pairs of product descriptions with label
class SiameseProductDataset(Dataset):


    #Initialise the dataset
    def __init__(self, table_a: pd.DataFrame, table_b: pd.DataFrame,
                 pairs_df: pd.DataFrame, text_column: str = 'combined_text'):

        self.table_a = table_a
        self.table_b = table_b
        self.pairs_df = pairs_df
        self.text_column = text_column

        #Validate inputs
        required_cols = ['idx_a', 'idx_b', 'label']
        for col in required_cols:
            if col not in pairs_df.columns:
                raise ValueError(f"Pairs DataFrame must contain '{col}' column")

    #Return the number of pairs in the dataset
    def __len__(self) -> int:

        return len(self.pairs_df)

    #Get a single pair from the dataset
    def __getitem__(self, idx: int) -> Dict[str, Any]:

        pair = self.pairs_df.iloc[idx]

        #Get indices and label
        idx_a = int(pair['idx_a'])
        idx_b = int(pair['idx_b'])
        label = float(pair['label'])

        #Get text descriptions
        text_a = self.table_a.iloc[idx_a].get(self.text_column, '')
        text_b = self.table_b.iloc[idx_b].get(self.text_column, '')

        #Ensure strings and ensure they handle NaN values
        text_a = str(text_a) if not pd.isna(text_a) else ''
        text_b = str(text_b) if not pd.isna(text_b) else ''

        return {
            'text_a': text_a,
            'text_b': text_b,
            'label': torch.tensor(label, dtype=torch.float32)
        }

#Siamese Neural Network architecture for product matching
#Uses shared weights to process two inputs and compare their embeddings
class SiameseNetwork(nn.Module):



    #Initialise the Siamese Network.
    def __init__(self, embedding_dim: int = 384, hidden_dim: int = 256,
                 dropout_rate: float = 0.3):

        super(SiameseNetwork, self).__init__()

        #Network architecture
        self.fc1 = nn.Linear(embedding_dim * 2, hidden_dim) #concatenated embeddings
        self.bn1 = nn.BatchNorm1d(hidden_dim) #Batch normalisation
        self.dropout1 = nn.Dropout(dropout_rate) #Regularisation

        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
        self.dropout2 = nn.Dropout(dropout_rate / 2)

        self.fc3 = nn.Linear(hidden_dim // 2, 1) #Output the similarity score

        #Initialise weights
        self._initialise_weights()

    #Initialise network weights using Xavier initialisation
    def _initialise_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight) #Xavier weight initialisation
                if module.bias is not None:
                    nn.init.zeros_(module.bias) #Zero bias inititalisation

    #Forward pass through the network and returns similarity score between 0 and 1 (batch_size,)
    def forward(self, embedding_a: torch.Tensor, embedding_b: torch.Tensor) -> torch.Tensor:
        #Concatenate embeddings from both products
        combined = torch.cat([embedding_a, embedding_b], dim=1)

        #Feed through network with ReLU activations
        x = F.relu(self.bn1(self.fc1(combined)))
        x = self.dropout1(x)

        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)

        #Output layer with sigmoid activation
        output = torch.sigmoid(self.fc3(x))

        #Always return 1D tensor
        output = output.squeeze()  # Remove size-1 dimensions

        #Handle edge case of if batch_size=1, squeeze makes it 0D
        if output.ndim == 0:
            output = output.unsqueeze(0)  # Make it 1D with size 1

        return output


#Siamese Neural Network with Sentence-BERT for product matching
#Implements deep learning approach with semantic text understanding
class SiameseProductMatcher:

    #Initialise the Siamese Network model
    def __init__(self, config: ProjectConfig):

        self.config = config
        self.sentence_model = None #Sentence-BERT model for embeddings
        self.siamese_net = None #Siamese neural network
        self.optimizer = None #Optimiser for training
        self.criterion = None #Loss function
        self.device = None #CPU or GPU devide
        self.is_trained = False #Training status flag
        self.training_started = False  # Track if training has started
        self.best_params = None #Store best hyperparameters found during tuning
        self.lr_scheduler = None #Learning rate scheduler

        #Initialise components
        self._initialise_model()

    def _initialise_model(self, learning_rate: float = None, hidden_dim: int = None, dropout_rate: float = 0.3):
        #Initialise all model components
        #Set device (GPU if available)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

        #Load pre-trained sentence transformer (the Sentence-BERT)
        print("Loading Sentence-BERT model...")
        self.sentence_model = SentenceTransformer(self.config.SENTENCE_TRANSFORMER_MODEL)
        self.sentence_model.to(self.device) #Move to appropriate device

        #Get embedding dimension from Sentence-BERT
        embedding_dim = self.sentence_model.get_sentence_embedding_dimension()

        #Use provided parameters or defaults from config
        if hidden_dim is None:
            hidden_dim = self.config.SIAMESE_HIDDEN_DIM

        if learning_rate is None:
            learning_rate = self.config.SIAMESE_LEARNING_RATE

        #Create Siamese Network
        self.siamese_net = SiameseNetwork(
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            dropout_rate=dropout_rate
        )
        self.siamese_net.to(self.device)

        #Initialise optimiser which is AdamW wih weight decay
        self.optimizer = AdamW(
            self.siamese_net.parameters(),
            lr=learning_rate,
            weight_decay=1e-4 #Regularisation
        )

        #Initialise learning rate scheduler
        self.lr_scheduler = ReduceLROnPlateau(
            self.optimizer,
            mode='max',  #We want to maximise F1-score
            factor=0.5,  #Reduce LR by half
            patience=3   #Number of epochs with no improvement
            #Removed 'verbose' parameter as it's not supported in some PyTorch versions
        )

        #Initialise loss function
        self.criterion = nn.BCELoss()


#Train for one epoch.
    def train_epoch(self, train_loader: DataLoader) -> float:
        self.siamese_net.train() #Set the model to training mode
        total_loss = 0.0
        n_batches = len(train_loader)

        progress_bar = tqdm(train_loader, desc="Training", leave=False)

        for batch_idx, batch in enumerate(progress_bar):
            #Move data to device
            texts_a = batch['text_a']
            texts_b = batch['text_b']
            labels = batch['label'].to(self.device)

            #Get embeddings from Sentence-BERT
            with torch.no_grad():
                embeddings_a = self.sentence_model.encode(
                    texts_a,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )
                embeddings_b = self.sentence_model.encode(
                    texts_b,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )

            #Forward pass through SIamese network
            self.optimizer.zero_grad() #Clear previous gradients
            outputs = self.siamese_net(embeddings_a, embeddings_b)
            loss = self.criterion(outputs, labels) #Calculate the loss

            #Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.siamese_net.parameters(), max_norm=1.0)  # Gradient clipping
            self.optimizer.step()

            #Update statistics
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        return total_loss / n_batches #The average loss per batch

    #Validate the model
    def validate(self, val_loader: DataLoader) -> Tuple[float, Dict[str, float]]:
        self.siamese_net.eval() #Set model to evaluation mode
        total_loss = 0.0
        all_predictions = []
        all_labels = []

        with torch.no_grad(): #No gradient calculation during validation
            progress_bar = tqdm(val_loader, desc="Validation", leave=False)

            for batch in progress_bar:
                #Move data to device
                texts_a = batch['text_a']
                texts_b = batch['text_b']
                labels = batch['label'].to(self.device)

                #Get embeddings from Sentence-BERT
                embeddings_a = self.sentence_model.encode(
                    texts_a,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )
                embeddings_b = self.sentence_model.encode(
                    texts_b,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )

                # Forward pass
                outputs = self.siamese_net(embeddings_a, embeddings_b)
                loss = self.criterion(outputs, labels)

                #Store predictions and labels
                all_predictions.extend(outputs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                #Update statistics
                total_loss += loss.item()
                progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        #Convert to numpy arrays
        predictions = np.array(all_predictions)
        labels = np.array(all_labels)

        #Calculate binary predictions using the 0.5 threshold
        binary_predictions = (predictions >= 0.5).astype(int)

        #Calculate comprehensive metrics
        metrics = {
            'loss': total_loss / len(val_loader), #Average Validation loss
            'accuracy': accuracy_score(labels, binary_predictions),
            'precision': precision_score(labels, binary_predictions, zero_division=0),
            'recall': recall_score(labels, binary_predictions, zero_division=0),
            'f1': f1_score(labels, binary_predictions, zero_division=0),
            'roc_auc': roc_auc_score(labels, predictions) if len(np.unique(labels)) > 1 else 0.5
        }

        return metrics['loss'], metrics

    #Train the Siamese Network
    def train(self, train_dataset: SiameseProductDataset,
              val_dataset: SiameseProductDataset,
              epochs: Optional[int] = None,
              batch_size: Optional[int] = None,
              learning_rate: Optional[float] = None,
              hidden_dim: Optional[int] = None,
              dropout_rate: Optional[float] = None) -> pd.DataFrame:

        print(" Training Siamese Network...")

        #Use provided parameters or config values
        if epochs is None:
            epochs = self.config.SIAMESE_EPOCHS
        if batch_size is None:
            batch_size = self.config.SIAMESE_BATCH_SIZE
        if learning_rate is not None or hidden_dim is not None or dropout_rate is not None:
            #Reinitialise model with new hyperparameters
            self._initialise_model(
                learning_rate=learning_rate,
                hidden_dim=hidden_dim,
                dropout_rate=dropout_rate
            )

        #Create data loaders for training and validation
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True, #Shuffle training data
            num_workers=0  # Set to 0 for Colab compatibility
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False, #No need to shuffle validation data
            num_workers=0
        )

        print(f"Epochs: {epochs}")
        print(f"Batch size: {batch_size}")
        print(f"Learning rate: {self.optimizer.param_groups[0]['lr']}")
        print(f"Hidden dimension: {self.siamese_net.fc1.out_features}")
        print(f"Dropout rate: {self.siamese_net.dropout1.p}")
        print(f"Training samples: {len(train_dataset)}")
        print(f"Validation samples: {len(val_dataset)}")

        #Training history
        history = []
        best_val_f1 = 0.0 #Track the best F1-score for early stoppage
        best_model_state = None #Store best models weights
        patience_counter = 0 #For early stopping
        max_patience = 7 #Stop if no improvement for 7 epochs

        #Training loop over epochs
        for epoch in range(1, epochs + 1):
            print(f"\n Epoch {epoch}/{epochs}")

            #Training phase
            train_loss = self.train_epoch(train_loader)
            print(f"Training loss: {train_loss:.4f}")

            #Validate phase
            val_loss, val_metrics = self.validate(val_loader)
            print(f"Validation loss: {val_loss:.4f}")
            print(f"Validation accuracy: {val_metrics['accuracy']:.4f}")
            print(f"Validation F1-score: {val_metrics['f1']:.4f}")
            print(f"Validation AUC-ROC: {val_metrics['roc_auc']:.4f}")

            #Update learning rate based on validation F1
            old_lr = self.optimizer.param_groups[0]['lr']
            self.lr_scheduler.step(val_metrics['f1'])
            new_lr = self.optimizer.param_groups[0]['lr']

            if new_lr != old_lr:
                print(f"Learning rate updated: {old_lr:.2e} -> {new_lr:.2e}")

            #Save best model state
            if val_metrics['f1'] > best_val_f1:
                best_val_f1 = val_metrics['f1']
                best_model_state = {
                    'siamese_state_dict': self.siamese_net.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'epoch': epoch,
                    'val_f1': best_val_f1,
                    'val_metrics': val_metrics
                }
                print(f"Found better model (F1: {best_val_f1:.4f})")
                patience_counter = 0  #Reset patience counter
            else:
                patience_counter += 1
                print(f"No improvement for {patience_counter} epoch(s)")

            #Record history for this epoch
            history.append({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                **val_metrics
            })

            #Early stopping
            if patience_counter >= max_patience:
                print(f"\n Early stopping triggered after {epoch} epochs")
                break

        #Mark as trained so training process is completed
        self.training_started = True

        #Load best model if found
        if best_model_state is not None:
            print(f"\n Loading best model from epoch {best_model_state['epoch']} (F1: {best_model_state['val_f1']:.4f})")
            self.siamese_net.load_state_dict(best_model_state['siamese_state_dict'])
            self.optimizer.load_state_dict(best_model_state['optimizer_state_dict'])
            self.is_trained = True

            #Save the best model
            self.save_model(os.path.join(config.MODELS_PATH, 'siamese_best.pth'))
        else:
            print("Warning: No model improvement during training")
            #Mark as trained even if no improvement (at least it completed training)
            self.is_trained = True

        print(f"\n Siamese Network training complete!")
        print(f"Best validation F1-score: {best_val_f1:.4f}")

        return pd.DataFrame(history) #Return training history as DataFrame


    #Tune hyperparameters for Siamese Network using Bayesian optimisation
    def tune_hyperparameters(self, train_dataset: SiameseProductDataset,
                           val_dataset: SiameseProductDataset,
                           param_space: Optional[Dict] = None) -> Dict[str, Any]:

        print("\n Tuning Siamese Network hyperparameters...")
        print(" Using Bayesian optimisation for efficient search...")

        #Define parameter space if not provided
        if param_space is None:
            param_space = {
                'learning_rate': (1e-6, 1e-3, 'log-uniform'),  #Log scale for learning rate
                'hidden_dim': (256, 1024, 'uniform'),          #Hidden layer dimension
                'dropout_rate': (0.1, 0.5, 'uniform'),         #Dropout rate
                'batch_size': [32, 64, 128, 256],              #Batch size options
                'weight_decay': (1e-5, 1e-2, 'log-uniform')    #Weight decay for AdamW
            }

        #Store original parameters
        original_params = {
            'learning_rate': self.config.SIAMESE_LEARNING_RATE,
            'hidden_dim': self.config.SIAMESE_HIDDEN_DIM,
            'batch_size': self.config.SIAMESE_BATCH_SIZE
        }

        #Import optuna for Bayesian optimisation
        import optuna
        print("Using Optuna for Bayesian optimisation...")

        def objective(trial):
            #Suggest hyperparameters
            lr = trial.suggest_float('learning_rate', 1e-6, 1e-3, log=True)
            hidden_dim = trial.suggest_int('hidden_dim', 256, 1024)
            dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
            batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
            weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

            print(f"\n Trial {trial.number}: LR={lr:.2e}, Hidden={hidden_dim}, Dropout={dropout_rate:.2f}, BS={batch_size}, WD={weight_decay:.2e}")

            #Create temporary model with suggested parameters
            temp_model = SiameseProductMatcher(self.config)
            temp_model._initialise_model(
                learning_rate=lr,
                hidden_dim=hidden_dim,
                dropout_rate=dropout_rate
            )

            #Set weight decay
            for param_group in temp_model.optimizer.param_groups:
                param_group['weight_decay'] = weight_decay

            #Train for a few epochs
            history = temp_model.train(
                train_dataset,
                val_dataset,
                epochs=self.config.SIAMESE_TUNE_EPOCHS,
                batch_size=batch_size,
                learning_rate=lr,
                hidden_dim=hidden_dim,
                dropout_rate=dropout_rate
            )

            #Get best validation F1 from history
            best_f1 = history['f1'].max()

            #Clean up to free memory
            del temp_model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

            return best_f1

        #Create study and optimise
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=self.config.RANDOM_STATE),
            pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)
        )

        study.optimize(
            objective,
            n_trials=self.config.SIAMESE_TUNE_TRIALS,
            timeout=self.config.SIAMESE_TUNE_TIMEOUT,
            show_progress_bar=True
        )

        #Get best parameters
        best_params = study.best_params
        self.best_params = best_params

        print(f"\n Hyperparameter tuning complete!")
        print(f"Best trial: {study.best_trial.number}")
        print(f"Best validation F1: {study.best_value:.4f}")
        print(f"Best parameters: {best_params}")

        #Visualise optimisation history
        try:
            print("Generating optimisation history visualisation...")

            #Get trial data for plotting
            trials_df = study.trials_dataframe()

            if len(trials_df) > 1:
                #Create optimisation history plot using matplotlib
                fig, ax = plt.subplots(figsize=(10, 6))

                #Plot F1-score for each trial
                ax.plot(trials_df['number'], trials_df['value'], marker='o', linewidth=2, color='blue', label='Trial F1-score')

                #Add best value line
                best_value = trials_df['value'].max()
                ax.axhline(y=best_value, color='red', linestyle='--', linewidth=2, label=f'Best F1: {best_value:.4f}')

                #Customise plot
                ax.set_xlabel('Trial Number', fontsize=12)
                ax.set_ylabel('Validation F1-Score', fontsize=12)
                ax.set_title('Bayesian Optimisation History - Siamese Network', fontsize=16, fontweight='bold')
                ax.legend(loc='best')
                ax.grid(True, alpha=0.3)

                #Save figure to visualisations folder
                filename = "siamese_hyperparameter_optimisation.png"
                filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
                plt.savefig(filepath, dpi=300, bbox_inches='tight')
                print(f"  Saved optimisation history plot to: {filepath}")

                #Display the plot
                plt.show()

                #Create parameter importance visualisation if we have enough trials
                if len(trials_df) >= 5:
                    try:
                        #Create parameter importance plot
                        fig2, ax2 = plt.subplots(figsize=(10, 6))

                        #Extract parameter importance manually
                        param_names = ['learning_rate', 'hidden_dim', 'dropout_rate', 'batch_size', 'weight_decay']
                        importance_scores = []

                        for param in param_names:
                            if param in trials_df.columns:
                                #Simple importance measure: correlation with F1-score
                                correlation = trials_df[param].corr(trials_df['value'])
                                if not np.isnan(correlation):
                                    importance_scores.append((param, abs(correlation)))

                        if importance_scores:
                            #Sort by importance
                            importance_scores.sort(key=lambda x: x[1], reverse=True)
                            params, scores = zip(*importance_scores)

                            #Create bar chart
                            bars = ax2.barh(range(len(params)), scores, color='steelblue', alpha=0.8)

                            #Customise plot
                            ax2.set_yticks(range(len(params)))
                            ax2.set_yticklabels(params, fontsize=10)
                            ax2.set_xlabel('Parameter Importance (Absolute Correlation)', fontsize=12)
                            ax2.set_title('Hyperparameter Importance - Siamese Network', fontsize=16, fontweight='bold')
                            ax2.grid(True, axis='x', alpha=0.3)

                            #Add value labels
                            for i, bar in enumerate(bars):
                                width = bar.get_width()
                                ax2.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                                       f'{width:.3f}', ha='left', va='center', fontsize=9)

                            #Save figure
                            filename2 = "siamese_parameter_importance.png"
                            filepath2 = os.path.join(self.config.VISUALISATIONS_PATH, filename2)
                            plt.savefig(filepath2, dpi=300, bbox_inches='tight')
                            print(f"Saved parameter importance plot to: {filepath2}")

                            plt.show()
                        else:
                            print("Not enough data for parameter importance analysis")

                    except Exception as e:
                        print(f"Could not generate parameter importance plot: {str(e)}")
            else:
                print("Not enough trials to generate visualisation")

        except Exception as e:
            print(f"Could not generate optimisation visualisation: {str(e)}")
            import traceback
            traceback.print_exc()

        #Save tuning results
        tuning_path = os.path.join(self.config.RESULTS_PATH, 'siamese_tuning_results.json')

        #Convert to JSON serialisable format
        serialisable_results = {
            'best_params': best_params,
            'best_score': float(study.best_value),
            'best_trial': int(study.best_trial.number),
            'n_trials_completed': len(study.trials),
            'config': {
                'SIAMESE_TUNE_TRIALS': self.config.SIAMESE_TUNE_TRIALS,
                'SIAMESE_TUNE_TIMEOUT': self.config.SIAMESE_TUNE_TIMEOUT,
                'SIAMESE_TUNE_EPOCHS': self.config.SIAMESE_TUNE_EPOCHS
            }
        }

        #Add trial details if available
        try:
            trials_df = study.trials_dataframe()
            #Convert Timestamp columns to string for JSON serialization
            for col in trials_df.columns:
                if pd.api.types.is_datetime64_any_dtype(trials_df[col]):
                    trials_df[col] = trials_df[col].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
            #Convert to dictionary for JSON serialisation
            serialisable_results['trials'] = trials_df.to_dict('records')
        except Exception as e:
            serialisable_results['trials'] = []
            print(f"Note: Could not save trial details: {str(e)}")

        with open(tuning_path, 'w') as f:
            json.dump(serialisable_results, f, indent=2, default=str)

        print(f"Tuning results saved to: {tuning_path}")

        return {
            'best_params': best_params,
            'best_score': study.best_value,
            'study': study,
            'trials': study.trials_dataframe()
        }

    #Make predictions for text pairs and return array of predicitons
    def predict(self, texts_a: List[str], texts_b: List[str],
                batch_size: int = 32) -> np.ndarray:

        if not self.is_trained:
            raise ValueError("Model must be trained before making predictions")

        self.siamese_net.eval() #Set to evaluation mode
        all_predictions = []

        with torch.no_grad(): #No gradient calculation during prediction
            #Process in batches
            for i in tqdm(range(0, len(texts_a), batch_size), desc="Predicting"):
                batch_a = texts_a[i:i+batch_size]
                batch_b = texts_b[i:i+batch_size]

                #Get embeddings from Sentence-BERT
                embeddings_a = self.sentence_model.encode(
                    batch_a,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )
                embeddings_b = self.sentence_model.encode(
                    batch_b,
                    convert_to_tensor=True,
                    device=self.device,
                    show_progress_bar=False
                )

                #Make predictions through the Siamese network
                predictions = self.siamese_net(embeddings_a, embeddings_b)

                #Handle both scalar and array outputs
                pred_numpy = predictions.cpu().numpy()

                #If predictions is a scalar (0-d array), convert to 1D array
                if pred_numpy.ndim == 0:
                    all_predictions.append(pred_numpy.item())
                else:
                    all_predictions.extend(pred_numpy)

        return np.array(all_predictions)


    #Evaluate model on test dataset
    def evaluate(self, test_dataset: SiameseProductDataset) -> Dict[str, Any]:

        print("\n Evaluating Siamese Network on test set...")

        #Check if model is trained
        if not self.is_trained:
            print("Model not trained. Cannot evaluate.")
            return {
                'accuracy': 0.0,
                'precision': 0.0,
                'recall': 0.0,
                'f1': 0.0,
                'roc_auc': 0.5
            }

        #Get all texts and labels from test dataset
        texts_a = []
        texts_b = []
        true_labels = []

        for i in range(len(test_dataset)):
            sample = test_dataset[i]
            texts_a.append(sample['text_a'])
            texts_b.append(sample['text_b'])
            true_labels.append(sample['label'].item())

        #Get predictions for all test pair
        predictions = self.predict(texts_a, texts_b)
        binary_predictions = (predictions >= 0.5).astype(int)

        #Calculate comprehensive metrics
        metrics = {
            'accuracy': accuracy_score(true_labels, binary_predictions),
            'precision': precision_score(true_labels, binary_predictions, zero_division=0),
            'recall': recall_score(true_labels, binary_predictions, zero_division=0),
            'f1': f1_score(true_labels, binary_predictions, zero_division=0),
            'roc_auc': roc_auc_score(true_labels, predictions) if len(np.unique(true_labels)) > 1 else 0.5,
            'predictions': predictions, #Raw probabilities
            'binary_predictions': binary_predictions #Binary decisions
        }

        #Print results
        print(f"\n TEST SET PERFORMANCE:")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1-Score:  {metrics['f1']:.4f}")
        print(f"AUC-ROC:   {metrics['roc_auc']:.4f}")

        #Classification report
        print("\n CLASSIFICATION REPORT:")
        print(classification_report(true_labels, binary_predictions, target_names=['Non-Match', 'Match']))

        return metrics

    #Save trained model
    def save_model(self, filepath: str):

        if not self.is_trained:
            print(" Warning: Model may not be fully trained, but saving anyway for checkpointing")

        #Create directory if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)

        #Save model state
        torch.save({
            'siamese_state_dict': self.siamese_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'sentence_model_name': self.config.SENTENCE_TRANSFORMER_MODEL,
            'config': self.config,
            'is_trained': self.is_trained,
            'best_params': self.best_params
        }, filepath)

        print(f"Siamese Network saved to: {filepath}")

    #Load the model
    def load_model(self, filepath: str):

        #Load checkpoint
        checkpoint = torch.load(filepath, map_location=self.device)

        #Re-initialise model if needed
        if self.siamese_net is None:
            self._initialise_model()

        #Load state dictionaries
        self.siamese_net.load_state_dict(checkpoint['siamese_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.is_trained = checkpoint.get('is_trained', True)
        self.best_params = checkpoint.get('best_params', None)

        print(f"Siamese Network loaded from: {filepath}")

#Initialise Siamese Network model
siamese_model = SiameseProductMatcher(config)
print("Siamese Network model initialised successfully!")

Evaluation and Visualisation class

In [ ]:
#Handles model evaluation, comparison, and visualisation
#Produces comprehensive reports and visualisations for analysis
class ModelEvaluator:

    #Initialise the evaluator
    def __init__(self, config: ProjectConfig):

        self.config = config

        #Set visualisation style
        plt.style.use('seaborn-v0_8-darkgrid')
        sns.set_palette("husl")

    #Plot confusion matrix
    def plot_confusion_matrix(self, y_true: np.ndarray, y_pred: np.ndarray,
                             model_name: str, save: bool = True):

        fig, ax = plt.subplots(figsize=(8, 6))

        #Create confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Match', 'Match'])
        disp.plot(cmap='Blues', ax=ax, values_format='d') #Uses blues colourmap

        #Customise plot
        ax.set_title(f'Confusion Matrix - {model_name}', fontsize=16, fontweight='bold')
        ax.set_xlabel('Predicted Label', fontsize=12)
        ax.set_ylabel('True Label', fontsize=12)

        plt.tight_layout()

        #Save figure
        if save:
            filename = f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved confusion matrix to: {filepath}")

        plt.show()

    #Plot ROC curve
    def plot_roc_curve(self, y_true: np.ndarray, y_scores: np.ndarray,
                      model_name: str, save: bool = True):

        fig, ax = plt.subplots(figsize=(8, 6))

        #Calculate ROC curve points and AUC
        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        #Plot ROC curve
        ax.plot(fpr, tpr, color='darkorange', lw=2,
                label=f'ROC curve (AUC = {roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') #A random classifier line

        #Customise plot
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title(f'ROC Curve - {model_name}', fontsize=16, fontweight='bold')
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        #Save figure
        if save:
            filename = f"roc_curve_{model_name.lower().replace(' ', '_')}.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved ROC curve to: {filepath}")

        plt.show()

        return roc_auc #Return the AUC score for reporting

    #Plot the precision recall curve
    def plot_precision_recall_curve(self, y_true: np.ndarray, y_scores: np.ndarray,
                                   model_name: str, save: bool = True):

        fig, ax = plt.subplots(figsize=(8, 6))

        #Calculate precision-recall curve
        precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
        avg_precision = np.mean(precision) #Average precision

        #Plot precision-recall curve
        ax.plot(recall, precision, color='blue', lw=2,
                label=f'Precision-Recall curve (AP = {avg_precision:.3f})')

        #Customise plot
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('Recall', fontsize=12)
        ax.set_ylabel('Precision', fontsize=12)
        ax.set_title(f'Precision-Recall Curve - {model_name}', fontsize=16, fontweight='bold')
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        #Save figure
        if save:
            filename = f"precision_recall_{model_name.lower().replace(' ', '_')}.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved precision-recall curve to: {filepath}")

        plt.show()

        return avg_precision #Return the average precision for reporting

    #Plot feature importance for Random Forest
    def plot_feature_importance(self, feature_importances: pd.DataFrame,
                               top_n: int = 15, save: bool = True):

        if feature_importances is None or feature_importances.empty:
            print("No feature importances to plot")
            return

        fig, ax = plt.subplots(figsize=(10, 8))

        #Get top N features sorted by importance
        top_features = feature_importances.head(top_n).sort_values('importance', ascending=True)

        #Create horizontal bar chart
        bars = ax.barh(range(len(top_features)), top_features['importance'],
                      color='steelblue', alpha=0.8)

        #Customise plot
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features['feature'], fontsize=10)
        ax.set_xlabel('Feature Importance', fontsize=12)
        ax.set_title(f'Top {top_n} Feature Importances (Random Forest)',
                    fontsize=16, fontweight='bold')
        ax.grid(True, axis='x', alpha=0.3)

        #Add value labels
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{width:.3f}', ha='left', va='center', fontsize=9)

        plt.tight_layout()

        #Save figure
        if save:
            filename = "feature_importance_random_forest.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved feature importance plot to: {filepath}")

        plt.show()

    #Plot the training history for Siamese Network
    def plot_training_history(self, history_df: pd.DataFrame, save: bool = True):

        if history_df is None or history_df.empty:
            print("No training history to plot")
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10)) #A 2x2 grid of subplots

        #Plot 1) Training and validation loss over epochs
        axes[0, 0].plot(history_df['epoch'], history_df['train_loss'],
                        label='Training Loss', marker='o', linewidth=2)
        axes[0, 0].plot(history_df['epoch'], history_df['val_loss'],
                        label='Validation Loss', marker='s', linewidth=2)
        axes[0, 0].set_xlabel('Epoch', fontsize=12)
        axes[0, 0].set_ylabel('Loss', fontsize=12)
        axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        #Plot 2) The validation accuracy over epochs
        axes[0, 1].plot(history_df['epoch'], history_df['accuracy'],
                       color='green', marker='o', linewidth=2)
        axes[0, 1].set_xlabel('Epoch', fontsize=12)
        axes[0, 1].set_ylabel('Accuracy', fontsize=12)
        axes[0, 1].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
        axes[0, 1].grid(True, alpha=0.3)

        #Plot 3) Validation F1-Score over epochs
        axes[1, 0].plot(history_df['epoch'], history_df['f1'],
                       color='orange', marker='o', linewidth=2)
        axes[1, 0].set_xlabel('Epoch', fontsize=12)
        axes[1, 0].set_ylabel('F1-Score', fontsize=12)
        axes[1, 0].set_title('Validation F1-Score', fontsize=14, fontweight='bold')
        axes[1, 0].grid(True, alpha=0.3)

        #Plot 4) Validation AUC-ROC over epochs
        axes[1, 1].plot(history_df['epoch'], history_df['roc_auc'],
                       color='red', marker='o', linewidth=2)
        axes[1, 1].set_xlabel('Epoch', fontsize=12)
        axes[1, 1].set_ylabel('AUC-ROC', fontsize=12)
        axes[1, 1].set_title('Validation AUC-ROC', fontsize=14, fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3)

        #The title
        plt.suptitle('Siamese Network Training History', fontsize=18, fontweight='bold', y=1.02)
        plt.tight_layout()

        #Save figure
        if save:
            filename = "siamese_training_history.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved training history plot to: {filepath}")

        plt.show()

    #Compare the multiple models and create visualisation
    def plot_model_comparison(self, model_results: Dict[str, Dict[str, float]],
                             save: bool = True) -> pd.DataFrame:

        print("\n MODEL COMPARISON:")
        print("="*60)

        #Create comparison DataFrame
        comparison_df = pd.DataFrame(model_results).T

        #Display comparison table
        display(comparison_df.round(4))

        #Create visualisation
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1'] #The 4key metrics

        for idx, metric in enumerate(metrics_to_plot):
            ax = axes[idx // 2, idx % 2] #Determine the subplot position

            #Get values for this metric
            values = comparison_df[metric].values
            model_names = comparison_df.index.tolist()

            # Create bar chart
            bars = ax.bar(model_names, values, color=['steelblue', 'darkorange'], alpha=0.8)

            #Customise plot
            ax.set_ylabel(metric.capitalize(), fontsize=12)
            ax.set_title(f'{metric.capitalize()} Comparison', fontsize=14, fontweight='bold')
            ax.set_ylim([0, 1.05]) #Metrics range fromn 0 to 1
            ax.grid(True, axis='y', alpha=0.3)

            #Add value labels
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=10)

        #The overall title
        plt.suptitle('Model Performance Comparison', fontsize=18, fontweight='bold', y=1.02)
        plt.tight_layout()

        #Save figure
        if save:
            filename = "model_comparison.png"
            filepath = os.path.join(self.config.VISUALISATIONS_PATH, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  Saved model comparison plot to: {filepath}")

        plt.show()

        return comparison_df #Return DataFrame so it can be used later on


    #Generate a comprehensive evaluation report
    def generate_comprehensive_report(self, model_results: Dict[str, Dict[str, float]],
                                    dataset_info: Dict[str, Any],
                                    config: ProjectConfig) -> Dict[str, Any]:

        print("\\n GENERATING COMPREHENSIVE EVALUATION REPORT...")

        # Convert model_results to serialisable format
        def clean_metrics(metrics_dict):
            """Remove numpy arrays from metrics dictionary."""
            clean_dict = {}
            for key, value in metrics_dict.items():
                if key not in ['predictions', 'binary_predictions', 'confusion_matrix']:
                    if hasattr(value, 'item'):  # Check if it's a numpy scalar
                        clean_dict[key] = value.item()
                    else:
                        clean_dict[key] = value
            return clean_dict

        #Clean model results so remove the non-serialisable elements
        cleaned_model_results = {}
        for model_name, metrics in model_results.items():
            cleaned_model_results[model_name] = clean_metrics(metrics)

        #Determine best model based on F1-score
        best_f1 = 0.0
        best_model = None
        for model_name, metrics in cleaned_model_results.items():
            if 'f1' in metrics and metrics['f1'] > best_f1:
                best_f1 = metrics['f1']
                best_model = model_name

        #Create report
        report = {
            'report_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'dataset_info': dataset_info,
            'configuration': {
                'random_state': config.RANDOM_STATE,
                'test_size': config.TEST_SIZE,
                'validation_size': config.VALIDATION_SIZE,
                'rf_n_estimators': config.RF_N_ESTIMATORS,
                'siamese_epochs': config.SIAMESE_EPOCHS,
                'sentence_transformer': config.SENTENCE_TRANSFORMER_MODEL
            },
            'model_performance': cleaned_model_results,
            'conclusions': {
                'best_model': best_model,
                'best_f1_score': best_f1,
                'performance_notes': 'Models evaluated on test set with threshold 0.5'
            }
        }

        print("Comprehensive report generated!")

        return report

#Initialise evaluator
evaluator = ModelEvaluator(config)
print("Model evaluator initialised successfully!")

Main Execution Pipeline

In [ ]:
#Main pipeline for the product matching project
#Executes all steps from data loading to model evaluation
def run_product_matching_pipeline():

    print("="*80)
    print(" STARTING PRODUCT MATCHING PIPELINE")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


    #STEP 1) Load and inspect the data
    print("STEP 1: LOADING AND INSPECTING DATA")
    print("-"*60)

    try:
        #Load data from the CSV files
        table_a, table_b = data_loader.load_product_tables()

        #Inspect data quality
        quality_report = data_loader.inspect_data_quality()

        #Display samples
        data_loader.display_sample_products(n_samples=3)

        #Get column statistics
        stats_df = data_loader.get_column_statistics()

    except Exception as e:
        print(f" Error in data loading: {str(e)}")
        print("Please ensure tableA.csv and tableB.csv are in the data folder.")
        return None

    #STEP 2) Create the labelled dataset
    print("\n STEP 2: CREATING LABELLED DATASET")
    print("-"*60)

    try:
        #Create labelled dataset - NOW returns 4 values including splits
        pairs_df, table_a_clean, table_b_clean, splits = dataset_creator.create_labelled_dataset(
            table_a, table_b
        )

        #Prepare data for Random Forest
        train_df = splits['train']
        val_df = splits['validation']
        test_df = splits['test']

        X_train = train_df.drop(['label', 'idx_a', 'idx_b'], axis=1, errors='ignore')
        y_train = train_df['label']
        X_val = val_df.drop(['label', 'idx_a', 'idx_b'], axis=1, errors='ignore')
        y_val = val_df['label']
        X_test = test_df.drop(['label', 'idx_a', 'idx_b'], axis=1, errors='ignore')
        y_test = test_df['label']

        print(f"\n Dataset preparation complete!")
        print(f"   Training features shape: {X_train.shape}")
        print(f"   Testing features shape: {X_test.shape}")

    except Exception as e:
        print(f"Error in dataset creation: {str(e)}")
        return None


    #STEP 3) Train the random forest model
    print("\n STEP 3: TRAINING RANDOM FOREST MODEL")
    print("-"*60)

    try:
        #Train Random Forest (features now validated in FeatureEngineer)
        print(f"  Training data shape: {X_train.shape}")
        rf_results = rf_model.train(X_train, y_train, X_val, y_val)

        #Evaluate on test set
        rf_test_metrics = rf_model.evaluate(X_test, y_test)

        #Save model
        rf_model.save_model(os.path.join(config.MODELS_PATH, 'random_forest_model.joblib'))

        print(f"\n Random Forest training complete!")

        #After training Random Forest, add cross-validation
        print("\n Performing cross-validation on Random Forest...")
        cv_results = rf_model.cross_validate(X_train, y_train, cv=5)

        #Save CV results
        with open(os.path.join(config.RESULTS_PATH, 'cross_validation_results.json'), 'w') as f:
            json.dump(cv_results, f, indent=2)

    except Exception as e:
        print(f" Error in Random Forest training: {str(e)}")
        return None

    #STEP 4) Train the siamese network model
    print("\n STEP 4: TRAINING SIAMESE NETWORK MODEL")
    print("-"*60)

    try:
        #Create PyTorch datasets for Siamese Network
        train_siamese_dataset = SiameseProductDataset(
            table_a_clean, table_b_clean, splits['train']
        )
        val_siamese_dataset = SiameseProductDataset(
            table_a_clean, table_b_clean, splits['validation']
        )
        test_siamese_dataset = SiameseProductDataset(
            table_a_clean, table_b_clean, splits['test']
        )

        #Hyperparameter tuning if enabled
        if config.TUNE_SIAMESE_NETWORK:
            print("\n Performing hyperparameter tuning for Siamese Network...")
            tuning_results = siamese_model.tune_hyperparameters(
                train_siamese_dataset,
                val_siamese_dataset
            )

            #Save tuning results
            tuning_path = os.path.join(config.RESULTS_PATH, 'siamese_tuning_results.json')
            #Convert to JSON serialisable format
            serialisable_results = {}
            for key, value in tuning_results.items():
                if key != 'study' and key != 'trials':  #Skip non-serialisable objects
                    if isinstance(value, pd.DataFrame):
                        #Convert Timestamp columns to string
                        df = value.copy()
                        for col in df.columns:
                            if pd.api.types.is_datetime64_any_dtype(df[col]):
                                df[col] = df[col].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
                        serialisable_results[key] = df.to_dict()
                    else:
                        serialisable_results[key] = value

            with open(tuning_path, 'w') as f:
                json.dump(serialisable_results, f, indent=2, default=str)
            print(f"Tuning results saved to: {tuning_path}")

            #Train final model with best parameters
            if siamese_model.best_params:
                print(f"\n Training final model with best hyperparameters: {siamese_model.best_params}")
                siamese_history = siamese_model.train(
                    train_siamese_dataset,
                    val_siamese_dataset,
                    learning_rate=siamese_model.best_params.get('learning_rate'),
                    hidden_dim=siamese_model.best_params.get('hidden_dim'),
                    dropout_rate=siamese_model.best_params.get('dropout_rate'),
                    batch_size=siamese_model.best_params.get('batch_size')
                )
            else:
                #Train with default parameters if no best params found
                siamese_history = siamese_model.train(
                    train_siamese_dataset,
                    val_siamese_dataset
                )
        else:
            #Train without tuning
            siamese_history = siamese_model.train(
                train_siamese_dataset,
                val_siamese_dataset
            )

        #Evaluate on test set
        siamese_test_metrics = siamese_model.evaluate(test_siamese_dataset)

        #Save model
        siamese_model.save_model(os.path.join(config.MODELS_PATH, 'siamese_model.pth'))

        print(f"\n Siamese Network training complete!")

    except Exception as e:
        print(f"Error in Siamese Network training: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

    #STEP 5) Evaluation and visualisation
    print("\n STEP 5: EVALUATION AND VISUALISATION")
    print("-"*60)

    try:
        #Prepare model results
        model_results = {
            'Random Forest': rf_test_metrics,
            'Siamese Network': siamese_test_metrics
        }

        #Generate visualisations for Random Forest
        print("\n Generating Random Forest visualisations...")

        #Get the predictions and probabilities for visualisation
        rf_predictions = rf_model.predict(X_test)
        rf_probabilities = rf_model.predict(X_test, return_proba=True)

        #Create visualisations
        evaluator.plot_confusion_matrix(y_test, rf_predictions, "Random Forest")
        evaluator.plot_roc_curve(y_test, rf_probabilities, "Random Forest")
        evaluator.plot_precision_recall_curve(y_test, rf_probabilities, "Random Forest")

        #Plot the feature importance if available
        if rf_model.feature_importances is not None:
            evaluator.plot_feature_importance(rf_model.feature_importances, top_n=15)

        #Generate visualisations for Siamese Network
        print("\n Generating Siamese Network visualisations...")

        #Use the  predictions from siamese_test_metrics
        evaluator.plot_confusion_matrix(
            y_test.values,
            siamese_test_metrics.get('binary_predictions', []),
            "Siamese Network"
        )
        evaluator.plot_roc_curve(
            y_test.values,
            siamese_test_metrics.get('predictions', []),
            "Siamese Network"
        )
        evaluator.plot_precision_recall_curve(
            y_test.values,
            siamese_test_metrics.get('predictions', []),
            "Siamese Network"
        )
        evaluator.plot_training_history(siamese_history)

        #Model comparison
        print("\n Comparing models...")
        comparison_df = evaluator.plot_model_comparison(model_results)

        #Generate comprehensive report
        dataset_info = {
            'table_a_samples': len(table_a),
            'table_b_samples': len(table_b),
            'total_pairs': len(pairs_df),
            'positive_pairs': int(pairs_df['label'].sum()),
            'negative_pairs': len(pairs_df) - int(pairs_df['label'].sum())
        }

        comprehensive_report = evaluator.generate_comprehensive_report(
            model_results, dataset_info, config
        )

        #Add hyperparameter tuning results to report
        comprehensive_report['hyperparameter_tuning'] = {
            'random_forest_tuned': config.TUNE_RANDOM_FOREST,
            'siamese_network_tuned': config.TUNE_SIAMESE_NETWORK,
            'random_forest_best_params': rf_model.best_params if hasattr(rf_model, 'best_params') else None,
            'siamese_network_best_params': siamese_model.best_params if hasattr(siamese_model, 'best_params') else None
        }

        print("\n Evaluation and visualisation complete!")

    except Exception as e:
        print(f"Error in evaluation: {str(e)}")
        #more details about the error
        import traceback
        traceback.print_exc()
        return None

    #STEP 6) Save the results and return them
    print("\n STEP 6: SAVING RESULTS")
    print("-"*60)

    try:
        #Helper function to convert numpy types to JSON serialisable
        def convert_to_serialisable(obj):
            #Convert numpy types and arrays to JSON serialisable format
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, dict):
                return {key: convert_to_serialisable(value) for key, value in obj.items()}
            elif isinstance(obj, list):
                return [convert_to_serialisable(item) for item in obj]
            elif isinstance(obj, (pd.DataFrame, pd.Series)):
                return obj.to_dict()
            else:
                return obj

        #Convert model_results to serialisable format
        serialisable_model_results = convert_to_serialisable(model_results)

        #Convert comprehensive_report to serialisable format
        serialisable_comprehensive_report = convert_to_serialisable(comprehensive_report)

        #Save dataset and splits to CSV files
        pairs_df.to_csv(os.path.join(config.DATA_PATH, 'product_pairs_dataset.csv'), index=False)
        table_a_clean.to_csv(os.path.join(config.DATA_PATH, 'table_a_clean.csv'), index=False)
        table_b_clean.to_csv(os.path.join(config.DATA_PATH, 'table_b_clean.csv'), index=False)

        #Save individual splits
        train_df.to_csv(os.path.join(config.DATA_PATH, 'train_split.csv'), index=False)
        val_df.to_csv(os.path.join(config.DATA_PATH, 'validation_split.csv'), index=False)
        test_df.to_csv(os.path.join(config.DATA_PATH, 'test_split.csv'), index=False)

        #Save model results as JSON
        with open(os.path.join(config.RESULTS_PATH, 'model_results.json'), 'w') as f:
            json.dump(serialisable_model_results, f, indent=2)

        #Save comprehensive report as JSON
        with open(os.path.join(config.RESULTS_PATH, 'comprehensive_report.json'), 'w') as f:
            json.dump(serialisable_comprehensive_report, f, indent=2)

        #Save comparison DataFrame
        comparison_df.to_csv(os.path.join(config.RESULTS_PATH, 'model_comparison.csv'), index=True)

        #Save training history
        siamese_history.to_csv(os.path.join(config.RESULTS_PATH, 'siamese_training_history.csv'), index=False)

        #Save hyperparameter information
        hyperparam_info = {
            'random_forest': {
                'tuned': config.TUNE_RANDOM_FOREST,
                'best_params': rf_model.best_params if hasattr(rf_model, 'best_params') else None,
                'default_params': {
                    'n_estimators': config.RF_N_ESTIMATORS,
                    'max_depth': config.RF_MAX_DEPTH,
                    'min_samples_split': config.RF_MIN_SAMPLES_SPLIT,
                    'min_samples_leaf': config.RF_MIN_SAMPLES_LEAF
                }
            },
            'siamese_network': {
                'tuned': config.TUNE_SIAMESE_NETWORK,
                'best_params': siamese_model.best_params if hasattr(siamese_model, 'best_params') else None,
                'default_params': {
                    'learning_rate': config.SIAMESE_LEARNING_RATE,
                    'hidden_dim': config.SIAMESE_HIDDEN_DIM,
                    'batch_size': config.SIAMESE_BATCH_SIZE,
                    'epochs': config.SIAMESE_EPOCHS
                }
            }
        }

        with open(os.path.join(config.RESULTS_PATH, 'hyperparameter_info.json'), 'w') as f:
            json.dump(hyperparam_info, f, indent=2)

        print(f"\n Results saved to:")
        print(f"Data: {config.DATA_PATH}")
        print(f"Models: {config.MODELS_PATH}")
        print(f"Results: {config.RESULTS_PATH}")
        print(f"Visualisations: {config.VISUALISATIONS_PATH}")

    except Exception as e:
        print(f"Warning: Error saving results: {str(e)}")
        import traceback
        traceback.print_exc()

    #Final summary
    print("\n" + "="*80)
    print(" PRODUCT MATCHING PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*80)
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("\n FINAL RESULTS SUMMARY:")
    print("-"*40)

    #Display final metrics
    for model_name, metrics in model_results.items():
        print(f"\n{model_name}:")
        print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
        print(f"Precision: {metrics.get('precision', 0):.4f}")
        print(f"Recall:    {metrics.get('recall', 0):.4f}")
        print(f"F1-Score:  {metrics.get('f1', 0):.4f}")
        print(f"AUC-ROC:   {metrics.get('roc_auc', 0):.4f}")

    #Display hyperparameter tuning results
    print("\n HYPERPARAMETER TUNING SUMMARY:")
    print("-"*40)
    if rf_model.best_params:
        print(f"Random Forest tuned: ✓")
        print(f"  Best parameters: {rf_model.best_params}")
    else:
        print(f"Random Forest tuned: ✗ (used default parameters)")

    if siamese_model.best_params:
        print(f"Siamese Network tuned: ✓")
        print(f"  Best parameters: {siamese_model.best_params}")
    else:
        print(f"Siamese Network tuned: ✗ (used default parameters)")

    #Determine and announce best model
    #Calculate which model performed best based on F1-score
    best_f1 = 0.0
    best_model_name = None

    for model_name, metrics in model_results.items():
        if metrics.get('f1', 0) > best_f1:
            best_f1 = metrics['f1']
            best_model_name = model_name

    print(f"\n BEST MODEL: {best_model_name} (F1-Score: {best_f1:.4f})")

    #Add conclusions to the comprehensive report for future reference
    comprehensive_report['conclusions'] = {
        'best_model': best_model_name,
        'best_f1_score': best_f1
    }

    #Return all the results
    return {
        'table_a': table_a,
        'table_b': table_b,
        'table_a_clean': table_a_clean,
        'table_b_clean': table_b_clean,
        'pairs_df': pairs_df,
        'splits': splits,
        'rf_model': rf_model,
        'siamese_model': siamese_model,
        'model_results': model_results,
        'comprehensive_report': comprehensive_report,
        'siamese_history': siamese_history,
        'comparison_df': comparison_df
    }


#Run the main pipeline
print("Ready to execute the product matching pipeline!")
print("To run the complete pipeline, execute the next cell.")

Execute pipline


In [ ]:

print("IMPORTANT: This will execute the complete product matching pipeline.")
print("Make sure you have tableA.csv and tableB.csv in your data folder.\n")

response = input("Do you want to proceed? (yes/no): ")

if response.lower() in ['yes', 'y']:
    print("\n" + "="*80)
    print("EXECUTING PIPELINE...")
    print("="*80)

    #Run the pipeline
    results = run_product_matching_pipeline()

    if results is not None:
        print("\n Pipeline execution completed successfully!")
        print("   Results have been saved to the Drive.")
    else:
        print("\n Pipeline execution failed. Check error messages above.")
else:
    print("\n Pipeline execution cancelled.")

Demontsrate predictions

In [ ]:
# Make predictions with the trained models

def demonstrate_predictions(results: Dict[str, Any]):

    print("="*80)
    print("DEMONSTRATING PREDICTIONS")
    print("="*80)

    if results is None:
        print("No results available. Please run the pipeline first.")
        return

    #Extract components from results
    table_a_clean = results['table_a_clean']
    table_b_clean = results['table_b_clean']
    test_df = results['splits']['test']
    rf_model = results['rf_model']
    siamese_model = results['siamese_model']

    #Re-initialise feature engineer
    feature_engineer = FeatureEngineer(config)

    #Select a few test examples
    n_examples = 5
    print(f"\nSelecting {n_examples} random test examples...\n")

    for i in range(min(n_examples, len(test_df))):
        #Get a test pair with indices and true label
        test_pair = test_df.iloc[i]
        idx_a = int(test_pair['idx_a'])
        idx_b = int(test_pair['idx_b'])
        true_label = test_pair['label']

        #Get products
        product_a = table_a_clean.iloc[idx_a]
        product_b = table_b_clean.iloc[idx_b]

        print(f"\n{'='*60}")
        print(f"EXAMPLE {i+1}:")
        print(f"{'='*60}")

        #Display product information
        print(f"\n PRODUCT A:")
        print(f"Title: {product_a.get('title', 'N/A')[:80]}...")
        print(f"Brand: {product_a.get('brand', 'N/A')}")
        print(f"Model: {product_a.get('modelno', 'N/A')}")
        print(f"Price: ${product_a.get('price', 'N/A')}")

        print(f"\n PRODUCT B:")
        print(f"Title: {product_b.get('title', 'N/A')[:80]}...")
        print(f"Brand: {product_b.get('brand', 'N/A')}")
        print(f"Model: {product_b.get('modelno', 'N/A')}")
        print(f"Price: ${product_b.get('price', 'N/A')}")

        print(f"\n ACTUAL: {'MATCH' if true_label == 1 else 'NON-MATCH'}")

        #Random Forest prediction
        #First need to create features for this pair
        rf_features = feature_engineer.create_features_for_pair(product_a, product_b)
        rf_features_df = pd.DataFrame([rf_features])

        #Make sure to drop the label column if it exists
        if 'label' in rf_features_df.columns:
            rf_features_df = rf_features_df.drop('label', axis=1)

        #Get prediction probability and binary decision
        rf_prob = rf_model.predict(rf_features_df, return_proba=True)
        #Handle if rf_prob is an array
        rf_prob = rf_prob[0] if isinstance(rf_prob, (list, np.ndarray)) and len(rf_prob) > 0 else rf_prob
        rf_pred = 1 if rf_prob >= 0.5 else 0

        print(f"\n RANDOM FOREST PREDICTION:")
        print(f"Prediction: {'MATCH' if rf_pred == 1 else 'NON-MATCH'}")
        print(f"Confidence: {rf_prob:.3f}")
        print(f"Correct: {'✓' if rf_pred == true_label else 'X'}")

        #Siamese Network prediction
        text_a = product_a.get('combined_text', '')
        text_b = product_b.get('combined_text', '')
        siamese_prob = siamese_model.predict([text_a], [text_b])
        #Handle if siamese_prob is an array
        siamese_prob = siamese_prob[0] if isinstance(siamese_prob, (list, np.ndarray)) and len(siamese_prob) > 0 else siamese_prob
        siamese_pred = 1 if siamese_prob >= 0.5 else 0

        print(f"\n SIAMESE NETWORK PREDICTION:")
        print(f"Prediction: {'MATCH' if siamese_pred == 1 else 'NON-MATCH'}")
        print(f"Confidence: {siamese_prob:.3f}")
        print(f"Correct: {'✓' if siamese_pred == true_label else 'X'}")

        #Agreement between models
        if rf_pred == siamese_pred:
            print(f"\n MODELS AGREE: {'MATCH' if rf_pred == 1 else 'NON-MATCH'}")
        else:
            print(f"\n MODELS DISAGREE:")
            print(f"Random Forest: {'MATCH' if rf_pred == 1 else 'NON-MATCH'}")
            print(f"Siamese Network: {'MATCH' if siamese_pred == 1 else 'NON-MATCH'}")

    print(f"\n{'='*80}")
    print("Demonstration complete!")
    print(f"{'='*80}")


#Check if we have results from pipeline execution
if 'results' in locals() and results is not None:
    demonstrate_predictions(results)
else:
    print("No results available. Please run the pipeline cell first")

Utility functions for custom predictions

In [ ]:
#Utility functions for custom predictions
#Predict whether two products match using trained models
def predict_product_pair(product_a_dict: Dict[str, Any], product_b_dict: Dict[str, Any],
                        rf_model_path: str = None, siamese_model_path: str = None) -> Dict[str, Any]:

    print("Making prediction for custom product pair...")

    #Convert dictionaries to pandas Series for compatibility
    product_a = pd.Series(product_a_dict)
    product_b = pd.Series(product_b_dict)

    #Load models if paths provided, else use in-memory models
    if rf_model_path and os.path.exists(rf_model_path):
        print("Loading Random Forest model...")
        rf_model_local = RandomForestProductMatcher(config)
        rf_model_local.load_model(rf_model_path)
    else:
        print("Using in-memory Random Forest model...")
        rf_model_local = rf_model if 'rf_model' in locals() else None

    if siamese_model_path and os.path.exists(siamese_model_path):
        print("Loading Siamese Network model...")
        siamese_model_local = SiameseProductMatcher(config)
        siamese_model_local.load_model(siamese_model_path)
    else:
        print("Using in-memory Siamese Network model...")
        siamese_model_local = siamese_model if 'siamese_model' in locals() else None

    #Preprocess text using the same preprocessor as in training
    print("Preprocessing text...")
    preprocessor = TextPreprocessor(config)

    #Create single-row DataFrames for preprocessing
    product_a_df = pd.DataFrame([product_a_dict])
    product_b_df = pd.DataFrame([product_b_dict])

    #Apply the same preprocessing pipeline used during training
    product_a_clean = preprocessor.preprocess_product_text(product_a_df).iloc[0]
    product_b_clean = preprocessor.preprocess_product_text(product_b_df).iloc[0]

    #Prepare results dictionary
    results = {
        'product_a': product_a_dict,
        'product_b': product_b_dict,
        'predictions': {}
    }

    #Random Forest prediction
    if rf_model_local and rf_model_local.is_trained:
        print("Making Random Forest prediction...")
        feature_engineer = FeatureEngineer(config)
        #Extract features using the same feature engineering as in training
        rf_features = feature_engineer.create_features_for_pair(product_a_clean, product_b_clean)
        rf_features_df = pd.DataFrame([rf_features])
        #Get prediction probability and binary decision
        rf_prob = rf_model_local.predict(rf_features_df, return_proba=True)[0]
        rf_pred = 1 if rf_prob >= 0.5 else 0

        results['predictions']['random_forest'] = {
            'prediction': 'Match' if rf_pred == 1 else 'Non-Match',
            'confidence': float(rf_prob), #Convert numpy float to Python float
            'binary_prediction': int(rf_pred)
        }

    #Siamese Network prediction
    if siamese_model_local and siamese_model_local.is_trained:
        print("Making Siamese Network prediction...")
        #Get preprocesssed text for embediing
        text_a = product_a_clean.get('combined_text', '')
        text_b = product_b_clean.get('combined_text', '')

        #Get predicition probability
        siamese_prob = siamese_model_local.predict([text_a], [text_b])[0]
        siamese_pred = 1 if siamese_prob >= 0.5 else 0

        results['predictions']['siamese_network'] = {
            'prediction': 'Match' if siamese_pred == 1 else 'Non-Match',
            'confidence': float(siamese_prob),
            'binary_prediction': int(siamese_pred)
        }

    print("Prediction complete!")
    return results

#Make batch predictions for multiple product pairs
def batch_predict_product_pairs(product_pairs: List[Tuple[Dict, Dict]],
                               model_type: str = 'both') -> List[Dict[str, Any]]:

    print(f"Making batch predictions for {len(product_pairs)} product pairs...")

    all_results = []

    #Process each pair in the batch
    for i, (product_a_dict, product_b_dict) in enumerate(product_pairs):
        print(f"Processing pair {i+1}/{len(product_pairs)}...")
        #Use predict_product_pair function
        result = predict_product_pair(product_a_dict, product_b_dict)
        all_results.append(result)

    print(f"Batch predictions complete!")
    return all_results



Project summary and export

In [ ]:
#Generate a comprehensive project summary
def generate_project_summary(results: Dict[str, Any] = None) -> Dict[str, Any]:

    print("GENERATING PROJECT SUMMARY...")

    #Initialise summary structure
    model_performance = {}
    best_model = "N/A"
    best_f1 = 0.0
    dataset_info = {
        'table_a_samples': "N/A",
        'table_b_samples': "N/A",
        'total_pairs': "N/A"
    }
    #Extract metrics from results if available
    if results is not None:
        try:
            #Extract model performance metrics
            if 'model_results' in results:
                for model_name, metrics in results['model_results'].items():
                    model_perf = {}
                    for metric_name, metric_value in metrics.items():
                        #Skip non-serialisable elements
                        if metric_name not in ['predictions', 'binary_predictions', 'confusion_matrix']:
                            try:
                              #Convert numpy types to Python native types
                                if hasattr(metric_value, 'item'):
                                    model_perf[metric_name] = float(metric_value.item())
                                else:
                                    model_perf[metric_name] = float(metric_value) if isinstance(metric_value, (int, float, np.number)) else metric_value
                            except:
                                model_perf[metric_name] = metric_value
                    model_performance[model_name] = model_perf

                    #Track the best model based on the F1-Score
                    if 'f1' in model_perf and model_perf['f1'] > best_f1:
                        best_f1 = model_perf['f1']
                        best_model = model_name

            #Get dataset information
            if 'pairs_df' in results and hasattr(results['pairs_df'], '__len__'):
                dataset_info['total_pairs'] = len(results['pairs_df'])

            #Try to get table sizes
            try:
                if 'table_a' in results and hasattr(results['table_a'], '__len__'):
                    dataset_info['table_a_samples'] = len(results['table_a'])
                if 'table_b' in results and hasattr(results['table_b'], '__len__'):
                    dataset_info['table_b_samples'] = len(results['table_b'])
            except:
                pass #Continue if table sizes unavailable

        except Exception as e:
            print(f"Warning extracting results: {str(e)}")

    #Create summary
    summary = {
        'results': {
            'best_model': best_model,
            'best_f1_score': best_f1,
            'model_performance': model_performance,
            'dataset_size': dataset_info  # Always a dictionary
        }
    }

    #Save summary to JSON file
    summary_path = os.path.join(config.RESULTS_PATH, 'project_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"Project summary saved to: {summary_path}")

    return summary


#Generate and display summary
print("="*80)
print("PROJECT SUMMARY")
print("="*80)

#Check if we have results
if 'results' in locals() and results is not None:
    summary = generate_project_summary(results)

    if 'results' in summary:
        print(f"\n BEST MODEL: {summary['results']['best_model']}")
        print(f"Best F1-Score: {summary['results']['best_f1_score']:.4f}")

        print(f"\n DATASET STATISTICS:")
        print(f"Table A samples: {summary['results']['dataset_size']['table_a_samples']}")
        print(f"Table B samples: {summary['results']['dataset_size']['table_b_samples']}")
        print(f"Total pairs: {summary['results']['dataset_size']['total_pairs']}")
else:
    summary = generate_project_summary()
    print("\n No execution results available. Run the pipeline for complete results.")


print("PRODUCT MATCHING PROJECT CODE COMPLETE")
